# Novelty-Driven Mean Reversion
### A Portfolio Sort Study of Earnings Call Text (FIN 285F)
**Authoritative steps** follow **`project_specification_rio_zac.pdf`**. This notebook mirrors the memo’s section order; cell headings below cite the PDF §. Where **`FINAL.csv`** lacks turn/component rows, we use **sentence-level** splits on `full_transcript_text` and note the gap explicitly.

### Project steps (PDF → notebook)

1. **§1 Introduction** — Semantic novelty vs a firm’s own past language; overreaction and mean reversion; CMN (2020) benchmark. *Context:* this header and §10.
2. **§2 Research question & hypotheses** — **H1:** long Q1 (low adjusted novelty) / short Q5 (high); **H2:** larger event reaction for Q5 then partial reversal **[+2, +5]**; **H3:** FinBERT positive tone × novelty (optional). *Notebook:* §7 (event paths), §6b (regressions / FF5); H3 when you add FinBERT sorts.
3. **§3 Data sources & definitions** — Returns (`ret_t-15` … `ret_t+15`, overnight), identifiers (`permno`), `full_transcript_text`, and **constructed variables**: embedding novelty, lexical baselines (TF‑IDF / BM25 in memo), salience, LLM gate, SUE, macro penalty, section/analyst/topic weights (staged as data allow). *Notebook:* §1 load; §2–§6 NLP and merges.
4. **§4 Data policy & governance** — Sample rules: earnings calls, non-missing **`permno`**, **`word_count` ≥ 100**, **≥ 2** prior transcripts, **non-missing returns on [−1, +5]**; winsorize returns and overnight at **1st / 99th**; no look-ahead in baselines. *Notebook:* §1b + `nh.apply_project_spec_sample_pipeline` (`SAMPLE_FLOW`).
5. **§5 Exploratory data analysis** — Summary stats, return and text distributions, time coverage. *Notebook:* §1 preview cell; §8 diagnostics (extend with memo-style tables as needed).
6. **§6 Portfolio construction** — Within-quarter **quintiles** on adjusted novelty; **long Q1 / short Q5**; **quarterly** rebalance and holding period per memo; equal-weight baseline. *Notebook:* §7 implements an **event-window / CAR-style supplement** (memo §6.4); full **calendar-time quarterly sort** is the next build to match §6.1–6.3 exactly.
7. **§7 Model specification** — Risk-adjust returns (e.g. **Fama–French 5-factor**), **SUE** and **`pre_drift`** controls, clustered inference. *Notebook:* §6b (`nh.fit_ols_cluster`, FF5).

**Appendices (memo)** — **A** salience clusters → `salience_dictionary.py`; **B** NLP parameters → embedding / threshold config cells; **C** variable definitions → `novelty_helpers.py`; **D** sample construction flow → §1b `SAMPLE_FLOW`.

### NLP execution order (implements §3.3 precision gate + macro)
1. **Lexical baseline (TF‑IDF)** — per-firm rolling profile (memo lexical robustness path; also supports interpretation).
2. **Semantic novelty** — encoder + distance to historical centroid on **turn-aligned** segments (memo primary: **nomic-embed-text-v1.5**; notebook default `all-MiniLM-L6-v2` until swapped).
3. **Salience ≥ 0.30** and **novelty ≥ 0.60** → **Ollama** verbatim §3.3 YES/NO prompt → **LLM = YES** required for gated rows.
4. **Cross-market penalty** — sigmoid in cluster frequency (**f₀=0.40**, **k=15**); notebook uses **quarter × cluster** as proxy until trailing **three-month** prior window is merged (memo §3.3 / §4.4).
5. **Adjusted novelty** → firm-event aggregation and within-quarter **quintiles** for sorts.

> **Embeddings:** Not FinBERT for novelty geometry; memo encoder is **Nomic**; **H3** uses FinBERT **tone** separately.
>
> **Dev runs:** `_DEV_TICKERS_RAW` in §0 → scoped `*__dev_*.pkl` caches; `None` = full panel.


## 0. Environment & Imports

In [1]:
# ══════════════════════════════════════════════════════════════════════
# Google Colab Setup
# ══════════════════════════════════════════════════════════════════════
# Run this cell on Colab. It:
#   1. Clones the project repo (code, helpers, salience dict, scripts)
#   2. Mounts Google Drive for large data files (FINAL.csv, speaker indices)
#   3. Installs pip dependencies
#   4. Configures Ollama fast-fail (not available on Colab)
# On a local machine this cell is a harmless no-op.

import sys, os, shutil

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL    = "https://github.com/zac-garland/wrds.git"
    REPO_DIR    = "/content/wrds"
    PROJECT_DIR = os.path.join(REPO_DIR, "final_project")
    DRIVE_FOLDER = "/content/drive/MyDrive/FIN285F-Project-Dataset"

    # ── 1. Clone the GitHub repo (code + helpers + scripts) ─────────
    if os.path.isdir(REPO_DIR):
        print(f"Repo already cloned at {REPO_DIR} — pulling latest...")
        os.system(f"cd {REPO_DIR} && git pull --ff-only 2>&1")
    else:
        print(f"Cloning {REPO_URL} ...")
        os.system(f"git clone {REPO_URL} {REPO_DIR} 2>&1")
    assert os.path.isdir(PROJECT_DIR), f"Clone failed — {PROJECT_DIR} not found"
    print(f"  Repo ready: {PROJECT_DIR}")

    # ── 2. Copy code files from repo to /content (working dir) ──────
    for fname in ["novelty_helpers.py", "salience_dictionary.py"]:
        src = os.path.join(PROJECT_DIR, fname)
        dst = os.path.join("/content", fname)
        if os.path.exists(src):
            shutil.copy2(src, dst)
            print(f"  Copied {fname} from repo")

    # Copy scripts/ directory (ensure_ollama.sh etc.)
    scripts_src = os.path.join(PROJECT_DIR, "scripts")
    scripts_dst = "/content/scripts"
    if os.path.isdir(scripts_src):
        if os.path.exists(scripts_dst):
            shutil.rmtree(scripts_dst)
        shutil.copytree(scripts_src, scripts_dst)
        print("  Copied scripts/ from repo")

    # ── 3. Mount Google Drive (large data files only) ────────────────
    from google.colab import drive
    drive.mount("/content/drive")
    assert os.path.isdir(DRIVE_FOLDER), (
        f"Drive folder not found: {DRIVE_FOLDER}\n"
        "Create 'FIN285F-Project-Dataset' in Google Drive and upload:\n"
        "  - FINAL.csv  (~10 GB)\n"
        "  - transcript_speaker_indices.csv"
    )

    # ── 4. Symlink large data files (avoids copying ~10 GB) ─────────
    for fname in ["FINAL.csv", "transcript_speaker_indices.csv"]:
        src = os.path.join(DRIVE_FOLDER, fname)
        dst = os.path.join("/content", fname)
        if os.path.islink(dst) or os.path.isfile(dst):
            os.remove(dst)
        if os.path.exists(src):
            os.symlink(src, dst)
            size_gb = os.path.getsize(src) / 1e9
            print(f"  Linked {fname} ({size_gb:.1f} GB)")
        else:
            print(f"  WARNING: {fname} not found in {DRIVE_FOLDER}")

    # ── 5. Symlink pkl_cache/ to Drive (persists across sessions) ───
    #    Saves HOURS on re-runs: baselines, novelty scores, LLM cache.
    cache_drive = os.path.join(DRIVE_FOLDER, "pkl_cache")
    cache_local = "/content/pkl_cache"
    os.makedirs(cache_drive, exist_ok=True)
    if os.path.islink(cache_local):
        os.remove(cache_local)
    elif os.path.isdir(cache_local):
        shutil.rmtree(cache_local)
    os.symlink(cache_drive, cache_local)
    print(f"  Linked pkl_cache/ -> Drive (caches persist across sessions)")

    # ── 6. Cache HuggingFace models on Drive ────────────────────
    #    Avoids re-downloading sentence-transformers model each session.
    hf_cache = os.path.join(DRIVE_FOLDER, "hf_cache")
    os.makedirs(hf_cache, exist_ok=True)
    os.environ["HF_HOME"] = hf_cache
    os.environ["SENTENCE_TRANSFORMERS_HOME"] = os.path.join(hf_cache, "sentence_transformers")
    print(f"  Linked HF model cache -> {hf_cache}")

    # ── 7. Install dependencies ─────────────────────────────────────
    print("\nInstalling packages...")
    os.system(
        "pip install -q sentence-transformers statsmodels tqdm "
        "yfinance pandas-datareader 2>/dev/null"
    )

    # ── 8. Ollama fast-fail (not available on Colab) ────────────────
    os.environ["OLLAMA_HTTP_TIMEOUT"] = "3"
    os.environ["OLLAMA_MAX_RETRIES"]  = "0"

    # ── 9. GPU check ────────────────────────────────────────────────
    try:
        import torch
        gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
    except Exception:
        gpu = "Unknown"

    print(f"\n{'='*55}")
    print(f"  Colab setup complete")
    print(f"  Repo:          {REPO_URL}")
    print(f"  Drive data:    {DRIVE_FOLDER}")
    print(f"  Working dir:   {os.getcwd()}")
    print(f"  GPU:           {gpu}")
    print(f"  Ollama:        DISABLED (fast-fail; salience-only)")
    print(f"{'='*55}")

else:
    print("Not on Colab — skipping setup. Running locally.")

Cloning https://github.com/zac-garland/wrds.git ...
  Repo ready: /content/wrds/final_project
  Copied novelty_helpers.py from repo
  Copied salience_dictionary.py from repo
  Copied scripts/ from repo
Mounted at /content/drive
  Linked FINAL.csv (7.2 GB)
  Linked transcript_speaker_indices.csv (0.6 GB)
  Linked pkl_cache/ -> Drive (caches persist across sessions)
  Linked HF model cache -> /content/drive/MyDrive/FIN285F-Project-Dataset/hf_cache

Installing packages...

  Colab setup complete
  Repo:          https://github.com/zac-garland/wrds.git
  Drive data:    /content/drive/MyDrive/FIN285F-Project-Dataset
  Working dir:   /content
  GPU:           None
  Ollama:        DISABLED (fast-fail; salience-only)


In [3]:
# ══════════════════════════════════════════════════════════════════════
# Ollama on Colab (optional — set ENABLE_OLLAMA = True to use LLM gate)
# ══════════════════════════════════════════════════════════════════════
# Installs Ollama, caches model weights on Drive (~4.7 GB for llama3.1),
# and starts the server in the background.
#
# Requirements: Colab with GPU runtime (T4 is fine — 8B model fits in 16 GB)
# First run: ~5 min (4.7 GB model download, cached to Drive after)
# Subsequent: ~30s (loads from Drive cache)
#
# Set to False to skip — the pipeline still works with salience-only signal.

ENABLE_OLLAMA = True   # ← flip to False if you don't need the LLM gate

import sys, os, subprocess, time, shutil, glob

if "google.colab" in sys.modules and ENABLE_OLLAMA:
    DRIVE_FOLDER = "/content/drive/MyDrive/FIN285F-Project-Dataset"

    # ── 1. Install system dependency (zstd) + Ollama binary ──────────
    OLLAMA_BIN = shutil.which("ollama")

    if OLLAMA_BIN is None:
        # Colab may be missing zstd which the Ollama installer now requires
        print("Installing zstd + Ollama...")
        os.system("apt-get update -qq && apt-get install -y -qq zstd > /dev/null 2>&1")

        # Official install script (now zstd is available)
        r = subprocess.run(
            ["bash", "-c", "curl -fsSL https://ollama.com/install.sh | sh"],
            capture_output=True, text=True, timeout=180,
        )
        if r.stdout:
            # Show last few meaningful lines
            lines = [l for l in r.stdout.strip().splitlines() if l.strip()]
            print("\n".join(lines[-5:]))
        if r.returncode != 0 and r.stderr:
            err = [l for l in r.stderr.splitlines()
                   if l.strip() and "%" not in l]
            if err:
                print("stderr:", "\n".join(err[-5:]))

        # Find the binary
        OLLAMA_BIN = shutil.which("ollama")
        if OLLAMA_BIN is None:
            for p in ["/usr/local/bin/ollama", "/usr/bin/ollama",
                      "/usr/local/lib/ollama/ollama",
                      os.path.expanduser("~/.ollama/ollama")]:
                if os.path.isfile(p) and os.access(p, os.X_OK):
                    OLLAMA_BIN = p
                    break

    if OLLAMA_BIN and os.path.isfile(OLLAMA_BIN):
        print(f"  Ollama binary: {OLLAMA_BIN}")
        v = subprocess.run([OLLAMA_BIN, "--version"],
                           capture_output=True, text=True, timeout=10)
        print(f"  Version: {(v.stdout or v.stderr).strip()}")
    else:
        print("ERROR: Could not install Ollama. LLM gate disabled.")
        print("  Debug — run in a new Colab cell:")
        print("    !apt-get install -y zstd")
        print("    !curl -fsSL https://ollama.com/install.sh | sh")
        print("    !which ollama || find / -name ollama -type f 2>/dev/null")
        ENABLE_OLLAMA = False

if "google.colab" in sys.modules and ENABLE_OLLAMA:
    # ── 2. Cache model weights on Drive (persist across sessions) ────
    ollama_models = os.path.join(DRIVE_FOLDER, "ollama_models")
    os.makedirs(ollama_models, exist_ok=True)
    os.environ["OLLAMA_MODELS"] = ollama_models
    print(f"  Model cache: {ollama_models}")

    # ── 3. Start ollama serve in background ──────────────────────────
    os.system("pkill -f 'ollama serve' 2>/dev/null")
    time.sleep(1)

    _env = {**os.environ, "OLLAMA_MODELS": ollama_models}
    subprocess.Popen(
        [OLLAMA_BIN, "serve"],
        stdout=open("/tmp/ollama_stdout.log", "w"),
        stderr=open("/tmp/ollama_stderr.log", "w"),
        env=_env,
    )
    print("  Started ollama serve (background)")

    # Wait for server to be ready
    import requests
    _server_ok = False
    for i in range(30):
        try:
            r = requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
            if r.status_code == 200:
                print(f"  Server ready (took {i+1}s)")
                _server_ok = True
                break
        except Exception:
            pass
        time.sleep(1)

    if not _server_ok:
        print("  WARNING: Ollama server did not start in 30s")
        print("  Check: !cat /tmp/ollama_stderr.log")
    else:
        # ── 4. Pull model (cached on Drive — fast after first time) ──
        model_name = os.environ.get("OLLAMA_MODEL", "llama3.1")
        print(f"  Pulling {model_name} (first time ~5 min, then cached)...")
        result = subprocess.run(
            [OLLAMA_BIN, "pull", model_name],
            capture_output=True, text=True, timeout=600,
            env=_env,
        )
        if result.returncode == 0:
            print(f"  Model ready: {model_name}")
        else:
            print(f"  Pull failed: {result.stderr[:300]}")

        # ── 5. Verify with a warm-up query ────────────────────────
        #    First inference after pull is slow (loads model into RAM/VRAM).
        #    Use a long timeout and retry once if needed.
        print("  Warm-up query (first call loads model — may take 1-2 min)...")
        _test_ok = False
        for _attempt in range(2):
            try:
                test = requests.post(
                    "http://127.0.0.1:11434/api/generate",
                    json={"model": model_name, "prompt": "Say OK",
                          "stream": False, "options": {"num_predict": 5}},
                    timeout=180,
                )
                if test.status_code == 200:
                    reply = test.json().get("response", "").strip()[:50]
                    print(f"  Test query OK: {reply}")
                    _test_ok = True
                    break
                else:
                    print(f"  Attempt {_attempt+1}: HTTP {test.status_code}")
            except requests.exceptions.ReadTimeout:
                print(f"  Attempt {_attempt+1}: timed out (model still loading), retrying...")
            except Exception as e:
                print(f"  Attempt {_attempt+1}: {e}")
                break

        if _test_ok:
            os.environ["OLLAMA_HTTP_TIMEOUT"] = "180"
            os.environ["OLLAMA_MAX_RETRIES"] = "3"
            print("  LLM intent filter: ENABLED")
        else:
            print("  Warm-up failed — LLM gate disabled (salience-only signal)")

    print()

elif "google.colab" in sys.modules:
    print("Ollama DISABLED (ENABLE_OLLAMA = False). Salience-only signal.")
else:
    print("Not on Colab — Ollama managed locally.")

Installing zstd + Ollama...
  Ollama binary: /usr/local/bin/ollama
  Version: Warning: could not connect to a running Ollama instance
  Model cache: /content/drive/MyDrive/FIN285F-Project-Dataset/ollama_models
  Started ollama serve (background)
  Server ready (took 2s)
  Pulling llama3.1 (first time ~5 min, then cached)...
  Model ready: llama3.1
  Warm-up query (first call loads model — may take 1-2 min)...


KeyboardInterrupt: 

In [ ]:
# Install dependencies if needed
# !pip install sentence-transformers scikit-learn pandas numpy statsmodels

import warnings
warnings.filterwarnings("ignore")

import sys, os, re, json, pickle
import importlib.util
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

# Sentence embeddings — semantic encoder, NOT FinBERT (sentiment classifier)
from sentence_transformers import SentenceTransformer

# Project root: cwd, cwd/final_project, or any parent/*/final_project (robust to Jupyter cwd).
def _final_project_root() -> Path:
    here = Path.cwd().resolve()
    for c in (here, here / "final_project"):
        if (c / "novelty_helpers.py").is_file():
            return c.resolve()
    for base in here.parents:
        fp = base / "final_project" / "novelty_helpers.py"
        if fp.is_file():
            return fp.parent.resolve()
    raise FileNotFoundError(
        "novelty_helpers.py not found — open a terminal in `final_project` or run Jupyter with cwd there."
    )


def _load_novelty_helpers(project_root: Path):
    """Load `novelty_helpers.py` by path so a different package named novelty_helpers cannot shadow it."""
    path = project_root / "novelty_helpers.py"
    spec = importlib.util.spec_from_file_location("novelty_helpers", path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Cannot load {path}")
    mod = importlib.util.module_from_spec(spec)
    sys.modules["novelty_helpers"] = mod
    spec.loader.exec_module(mod)
    return mod


_PROJECT_ROOT = _final_project_root()
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

# Salience dictionary — same directory as novelty_helpers.py
from salience_dictionary import score_sentence, SALIENCE_DICT

if "novelty_helpers" in sys.modules:
    del sys.modules["novelty_helpers"]
nh = _load_novelty_helpers(_PROJECT_ROOT)

# --- Semantic encoder: change only these when moving to Nomic (then delete/rename pickle caches) ---
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"  # later: e.g. "nomic-ai/nomic-embed-text-v1.5" if exposed via sentence-transformers
EMBEDDING_CACHE_TAG = "exec_novelty_chunk_text_v1"  # bump when novelty geometry/segmentation changes

# Dev: small ticker sample (e.g. 3 names) for fast runs. Uses *__dev_AAPL_MSFT_NVDA.pkl sidecars;
# falls back to slicing the full pickle if a dev file does not exist yet. None = full panel.
_DEV_TICKERS_RAW =  ("AAPL","MSFT","NVDA") # e.g. "NVDA" or ("AAPL","MSFT","NVDA"); None = all rows in top50 below
DEV_TICKERS = nh.dev_tickers_normalized(_DEV_TICKERS_RAW)
# After changing this or the ticker list, re-run: df → turn-index cell → baselines → novelty (pickles are dev-scoped).


def embedding_cache_path(stem: str) -> Path:
    """Pickle path keyed by EMBEDDING_CACHE_TAG so a new encoder does not clobber old caches."""
    cache_dir = Path("./pkl_cache")
    cache_dir.mkdir(parents=True, exist_ok=True)
    if EMBEDDING_CACHE_TAG == "default":
        return cache_dir / f"{stem}.pkl"
    return cache_dir / f"{stem}_{EMBEDDING_CACHE_TAG}.pkl"


print("All imports OK")
print(f"Salience clusters: {len(SALIENCE_DICT)}")
print(f"Embedding model: {EMBEDDING_MODEL_NAME} | cache tag: {EMBEDDING_CACHE_TAG}")
if DEV_TICKERS:
    print(f"DEV_TICKERS={DEV_TICKERS!r} — dev sample mode (scoped cache files)")


## 1. Data load — PDF §3 (sources & definitions)

**PDF §3.1–3.2:** financial fields (daily returns around the call, overnight return, `permno`, event dates) and **text** (`full_transcript_text`). **`FINAL.csv`** is the working extract (~181k calls). The next cell **previews** columns without loading full text; the following cell loads `USE_COLS` via `novelty_helpers` (**returns** `ret_t-15` … `ret_t15`, **I/B/E/S** / SUE, **Compustat** links, **`full_transcript_text`**).

**Not in file:** market capitalization (merge from CRSP/Compustat by `permno`/`gvkey` when you add size controls). **Turn metadata** is merged from **`transcript_speaker_indices.csv`** (same folder as this notebook).

**Vs. prior `aggregated_all_years_updated_event.csv`:** Same column names for every field that existed before; `FINAL.csv` only **adds** Compustat/I-B-E-S link columns (no renames, no dropped core fields). The event panel is **slightly smaller** (stricter sample / merge), so a handful of `transcriptid`s differ from the old file.

**Turn NLP caches:** With `EMBEDDING_CACHE_TAG = "turns"`, novelty/baseline pickles are **`novelty_*_turns.pkl`**; re-run the baseline + novelty cells after changing the ticker panel, `transcript_speaker_indices.csv`, or the encoder.

In [ ]:
# Preview FINAL.csv shape & key fields (excludes full_transcript_text for speed)
DATA_PATH = Path(".") / "FINAL.csv"
assert DATA_PATH.exists(), f"Data not found at {DATA_PATH.resolve()}"

RET_COLS = [f"ret_t{j}" for j in list(range(-15, 0)) + list(range(0, 16))]
all_cols = list(pd.read_csv(DATA_PATH, nrows=0).columns)
PREVIEW_COLS = [c for c in all_cols if c != "full_transcript_text"]
prev = pd.read_csv(DATA_PATH, usecols=PREVIEW_COLS, low_memory=False)

print("=== FINAL.csv (metadata / numeric columns only) ===")
print(f"Path:   {DATA_PATH.resolve()}")
print(f"Rows:   {len(prev):,}")
print(f"Cols:   {len(all_cols)} (including full_transcript_text)")
print(f"permno missing:     {prev['permno'].isna().sum():,}")
print(f"ibes_sue_eps nn:    {prev['ibes_sue_eps'].notna().sum():,}  (SUE)")
print(f"ibes_raw_surp nn:   {prev['ibes_raw_surp_eps'].notna().sum():,}")
print(f"ibes_mean_est nn:   {prev['ibes_mean_est_eps'].notna().sum():,}")
wc = prev["word_count"]
print(f"word_count min/median/max: {wc.min()} / {wc.median()} / {wc.max()}")

print("\n--- Column groups ---")
groups = {
    "Identifiers & event": ["transcriptid", "companyid", "headline", "transcriptcreationdate_utc",
                            "mostimportantdateutc", "companyname", "ticker", "event_type", "call_date", "actual_call_date"],
    "Returns & prices": ["permno", "close_price_call_day", "open_price_next_day", "close_to_open_return"] + RET_COLS,
    "Text size": ["transcript_length", "word_count"],
    "Compustat": ["gvkey", "fiscal_period_end", "report_date", "fiscal_year", "fiscal_quarter", "compustat_actual_revenue"],
    "I/B/E/S": [c for c in prev.columns if c.startswith("ibes_")],
}
for label, cols in groups.items():
    hit = [c for c in cols if c in prev.columns]
    if hit:
        print(f"{label}: {', '.join(hit)}")

print("\nNote: turn splits use transcript_speaker_indices.csv (fallback: rough sentence splitter).")
print("Note: Market cap is not in FINAL.csv; add via CRSP/Compustat merge on permno/gvkey for regressions.")

sample_cols = [c for c in ["ticker", "mostimportantdateutc", "permno", "word_count", "ibes_sue_eps",
                           "ibes_raw_surp_eps", "close_to_open_return", "ret_t0", "ret_t1"] if c in prev.columns]
print("\n--- First 3 rows (sample columns) ---")
print(prev[sample_cols].head(3).to_string(index=False))

In [ ]:
DATA_PATH = Path(".") / "FINAL.csv"
assert DATA_PATH.exists(), f"Data not found at {DATA_PATH.resolve()}"

print("Loading FINAL.csv via nh.load_and_clean_final_csv (column list in novelty_helpers.py)...")
raw = nh.load_and_clean_final_csv(DATA_PATH)
print(f"After load + basic text clean: {raw.shape}")


## 1b. Sample construction — PDF §4 (data policy)

**PDF §4.1–4.3:** **non-missing `permno`**, **`word_count` ≥ 100**, **≥ 2 prior transcripts** per firm (strictly earlier quarters), **non-missing returns on [−1, +5]** (`ret_t-1` … `ret_t5`); **winsorize** those dailies and **`close_to_open_return`** at **1st / 99th** (pooled on the kept sample). **`pre_drift`** and **`sue`** are prepared for §7 controls.

Also adds **`pre_drift`** = sum(`ret_t-15` … `ret_t-1`) when all fifteen are non-missing (otherwise NaN — does not drop rows). Renames **`sue`** ← `ibes_sue_eps` for regressions later.

In [ ]:
# Project-spec sample + winsorize + pre_drift + sue (implementation: novelty_helpers)
raw, SAMPLE_FLOW = nh.apply_project_spec_sample_pipeline(raw)
print(SAMPLE_FLOW.to_string(index=False))
print(f"\npre_drift non-missing: {raw['pre_drift'].notna().sum():,} / {len(raw):,}")
print(f"\nFinal panel for downstream filter: {raw.shape[0]:,} rows × {raw.shape[1]} cols")

In [ ]:
# Top-50 S&P 500 by index weight — heavy NLP scope
import yfinance as yf          # or use a hardcoded list if offline
try:
    import pandas_datareader as pdr
    sp500 = pd.read_html("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies")[0]
    top50_tickers = sp500["Symbol"].head(50).tolist()
except Exception:
    # Fallback hardcoded top-50 as of 2024 (by weight)
    print('using fallback top 50')
    top50_tickers = [
        "AAPL","MSFT","NVDA","AMZN","META","GOOGL","GOOG","BRK-B","LLY","AVGO",
        "JPM","TSLA","UNH","V","XOM","MA","JNJ","PG","HD","MRK","COST","ABBV",
        "BAC","CRM","CVX","WFC","AMD","NFLX","KO","PEP","ORCL","TMO","ACN","LIN",
        "MCD","ABT","PM","GE","IBM","QCOM","DHR","AMGN","CAT","GS","INTC","ISRG",
        "SPGI","VZ","BLK","SCHW"
    ]


df = raw[raw["ticker"].isin(top50_tickers)].copy()
df = df.sort_values(["ticker", "quarter"]).reset_index(drop=True)

if DEV_TICKERS:
    _n0 = len(df)
    df = df[df["ticker"].str.upper().isin(DEV_TICKERS)].copy()
    print(f"DEV_TICKERS filter: kept {len(df):,} / {_n0:,} rows for {DEV_TICKERS!r}")
    if df.empty:
        raise ValueError(f"{DEV_TICKERS!r} not in top-50 subset — check ticker or disable dev mode")

print(f"Top-50 corpus: {df.shape[0]:,} transcripts, {df['ticker'].nunique()} firms")
print(f"Date range: {df['quarter'].min()} → {df['quarter'].max()}")


## 2. Lexical baseline (TF-IDF) — PDF §3.3 lexical / Cohen et al. analog

Build a TF-IDF vocabulary profile for each firm using all transcripts *before* a given quarter. This defines what is **normal** for that firm linguistically.

The rolling approach means: for each firm-quarter observation, the baseline is everything the firm has said in prior quarters. We store top-N terms per firm-quarter as the baseline fingerprint.


In [ ]:
def clean_text(txt: str) -> str:
    """Strip operator boilerplate and safe-harbour preambles."""
    txt = re.sub(r"\[Operator Instructions\]", "", txt, flags=re.I)
    txt = re.sub(r"(?i)forward.looking statements.{0,200}filings\.", "", txt)
    txt = re.sub(r"\s+", " ", txt)
    return txt.strip()

def split_sentences(txt: str) -> list[str]:
    """Rough sentence splitter (fallback when turn metadata is missing)."""
    sents = re.split(r"(?<=[.!?])\s+(?=[A-Z\"'(\[])", txt)
    return [s.strip() for s in sents if len(s.strip()) > 20]

SPEAKER_INDICES_CSV = _PROJECT_ROOT / "transcript_speaker_indices.csv"

# Build raw per-turn table (source of truth) for the current panel
raw_turn_df = nh.build_raw_turn_table(df, SPEAKER_INDICES_CSV)
print(f"Raw turns: {len(raw_turn_df):,} rows across {raw_turn_df['transcriptid'].nunique():,} transcripts")

# Derive (A) analyst attention per call (no lookahead) and (B) exec-scored chunks
analyst_attn = nh.analyst_attention_features_per_call(raw_turn_df)
exec_chunks = nh.build_exec_chunks(raw_turn_df)
print(f"Exec chunks: {len(exec_chunks):,} rows across {exec_chunks['transcriptid'].nunique():,} transcripts")

# Join attention back onto df for later firm-quarter regressions (optional)
df = df.merge(analyst_attn, on='transcriptid', how='left')

# Rolling analyst attention by topic (trailing ~3 months, no lookahead)
# 1) attach call dates onto raw turns
_call_dates = df[['transcriptid','mostimportantdateutc']].drop_duplicates('transcriptid')
_call_dates['mostimportantdateutc'] = pd.to_datetime(_call_dates['mostimportantdateutc'], errors='coerce')
raw_turn_df = raw_turn_df.merge(_call_dates, on='transcriptid', how='left')

# 2) score analyst turns to get dominant cluster (use salience dictionary)
analyst_turns = raw_turn_df[raw_turn_df['speakertypename'].astype(str) == 'Analysts'].copy()
analyst_turns = analyst_turns.dropna(subset=['mostimportantdateutc','transcriptpersonname','turn_text'])
analyst_turns = analyst_turns[['transcriptid','mostimportantdateutc','transcriptpersonname','turn_text']].copy()

from tqdm.auto import tqdm
tqdm.pandas()

def _dominant_cluster(text: str):
    score, breakdown = score_sentence(str(text)[:2000])
    if not breakdown:
        return None
    return max(breakdown, key=breakdown.get)

if len(analyst_turns):
    analyst_turns['top_cluster'] = analyst_turns['turn_text'].progress_apply(_dominant_cluster)
    analyst_turns = analyst_turns.dropna(subset=['top_cluster'])
    # one row per (call, analyst, cluster)
    analyst_turns = analyst_turns.drop_duplicates(['transcriptid','transcriptpersonname','top_cluster'])
    analyst_turns = nh.rolling_unique_analysts_by_cluster(
        analyst_turns,
        date_col='mostimportantdateutc',
        cluster_col='top_cluster',
        analyst_col='transcriptpersonname',
        window_days=90,
        out_col='trailing_unique_analysts_90d',
    )
    analyst_roll = (
        analyst_turns.groupby(['transcriptid','top_cluster'], sort=False)['trailing_unique_analysts_90d']
        .max()
        .reset_index()
    )
else:
    analyst_roll = pd.DataFrame(columns=['transcriptid','top_cluster','trailing_unique_analysts_90d'])

# Fast lookup: transcriptid → list of exec chunk texts
CHUNKS_BY_TID = (
    exec_chunks.sort_values(['transcriptid','chunk_index'], kind='mergesort')
    # Score on full chunk (operator/analyst context + executive response)
    .groupby('transcriptid')['chunk_text']
    .apply(list)
    .to_dict()
)

# Parallel lookups for debugging / display
EXEC_TEXT_BY_TID = (
    exec_chunks.sort_values(['transcriptid','chunk_index'], kind='mergesort')
    .groupby('transcriptid')['exec_text']
    .apply(list)
    .to_dict()
)
CONTEXT_TEXT_BY_TID = (
    exec_chunks.sort_values(['transcriptid','chunk_index'], kind='mergesort')
    .groupby('transcriptid')['context_text']
    .apply(list)
    .to_dict()
)

# Mapping for: novelty on EXEC *turns* → salience/LLM on full chunk_text
# (exec_chunks has one row per executive answer; we keep the exec turn index.)
_exec_chunk_map = exec_chunks[['transcriptid','chunk_index','exec_turn_index','chunk_text','exec_text']].copy()
_exec_chunk_map['transcriptid'] = _exec_chunk_map['transcriptid'].astype(int)
_exec_chunk_map['chunk_index'] = _exec_chunk_map['chunk_index'].astype(int)
_exec_chunk_map['exec_turn_index'] = _exec_chunk_map['exec_turn_index'].astype(int)
_exec_chunk_map = _exec_chunk_map.sort_values(['transcriptid','chunk_index'], kind='mergesort')

# transcriptid → list of exec turn_texts (one per chunk)
EXEC_TURN_TEXT_BY_TID = _exec_chunk_map.groupby('transcriptid')['exec_text'].apply(list).to_dict()
# transcriptid → list of full chunk_texts aligned to exec turns
CHUNK_TEXT_BY_EXEC_TURN_BY_TID = _exec_chunk_map.groupby('transcriptid')['chunk_text'].apply(list).to_dict()
# transcriptid → list of chunk_index aligned to exec turns
CHUNK_INDEX_BY_EXEC_TURN_BY_TID = _exec_chunk_map.groupby('transcriptid')['chunk_index'].apply(list).to_dict()

# transcriptid → list of ALL raw turn texts (Operator/Analyst/Executive) in turn_index order
FULL_TURNS_BY_TID = (
    raw_turn_df.sort_values(['transcriptid','turn_index'], kind='mergesort')
    .groupby('transcriptid')['turn_text']
    .apply(list)
    .to_dict()
)


def transcript_segments(txt: str, transcriptid: int, *, min_chars: int = 20) -> list[str]:
    """Return exec-scored chunks (operator/analyst context + exec turn).

    Falls back to rough sentence split only if transcriptid has no chunks.
    """
    segs = CHUNKS_BY_TID.get(int(transcriptid))
    if segs:
        return [s for s in segs if len(str(s).strip()) > min_chars]
    return split_sentences(clean_text(str(txt)))


In [ ]:
# Build rolling per-firm TF-IDF baselines
# For each (ticker, quarter), fit TF-IDF on ALL prior quarters of that firm.

print("Building per-firm rolling TF-IDF baselines...")

MAIN_BASELINE_CACHE = embedding_cache_path("novelty_baselines")
BASELINE_CACHE = nh.dev_scoped_cache_path(MAIN_BASELINE_CACHE, DEV_TICKERS)
if DEV_TICKERS:
    print(f"  Baseline pickle: {BASELINE_CACHE.name} (dev) | main: {MAIN_BASELINE_CACHE.name}")

if BASELINE_CACHE.exists():
    with open(BASELINE_CACHE, "rb") as f:
        firm_baselines = pickle.load(f)
    print(f"Loaded cached baselines: {len(firm_baselines)} firm-quarter pairs")
elif DEV_TICKERS and MAIN_BASELINE_CACHE.exists():
    with open(MAIN_BASELINE_CACHE, "rb") as f:
        firm_baselines = nh.filter_baseline_dict(pickle.load(f), DEV_TICKERS)
    print(
        f"Loaded main cache + DEV filter: {len(firm_baselines)} keys for {DEV_TICKERS!r} "
        f"(from {MAIN_BASELINE_CACHE.name})"
    )
else:
    from tqdm.auto import tqdm
    firm_baselines = {}   # key: (ticker, quarter_str) → TfidfVectorizer fitted on prior docs

    for ticker, grp in tqdm(df.groupby("ticker"), desc="TF-IDF baselines", unit="firm"):
        quarters = sorted(grp["quarter_str"].unique())

        # Need at least 2 quarters of history before we can compute novelty
        for i, q in enumerate(quarters):
            if i < 2:
                continue   # skip first 2 quarters — not enough history

            prior_qs  = quarters[:i]
            prior_txt = grp[grp["quarter_str"].isin(prior_qs)]["full_transcript_text"].tolist()
            prior_txt = [clean_text(t) for t in prior_txt]

            if len(prior_txt) == 0:
                continue

            vec = TfidfVectorizer(
                max_features   = 3000,
                ngram_range    = (1, 2),
                min_df         = 1,
                stop_words     = "english",
                sublinear_tf   = True,
            )
            try:
                vec.fit(prior_txt)
                firm_baselines[(ticker, q)] = {
                    "vectorizer": vec,
                    "n_docs"    : len(prior_txt),
                    "vocab_size": len(vec.vocabulary_),
                }
            except Exception as e:
                pass

    with open(BASELINE_CACHE, "wb") as f:
        pickle.dump(firm_baselines, f)

    print(f"Built {len(firm_baselines)} firm-quarter baselines → wrote {BASELINE_CACHE.name}")

if nh.dev_mode_active(DEV_TICKERS):
    firm_baselines = nh.filter_baseline_dict(firm_baselines, DEV_TICKERS)
    print(f"  In-memory baselines after DEV guard: {len(firm_baselines)} keys")


In [ ]:
# Quick sanity check — inspect NVDA baseline
sample_key = [k for k in firm_baselines if k[0] == "NVDA"]
if sample_key:
    k = sorted(sample_key)[-1]
    bl = firm_baselines[k]
    print(f"NVDA baseline at {k[1]}:")
    print(f"  Trained on {bl['n_docs']} prior transcripts")
    print(f"  Vocabulary size: {bl['vocab_size']:,} terms")
    top_terms = sorted(bl['vectorizer'].vocabulary_.items(), key=lambda x: x[1], reverse=True)[:20]
    print(f"  Sample vocab: {[t for t,_ in top_terms[:20]]}")


## 3. Semantic novelty (embeddings) — PDF §3.3 primary path

For each firm-quarter, embed the **new** transcript **turns** (or sentence fallback) using **`EMBEDDING_MODEL_NAME`** (set in the imports cell) and compare them against the **historical segment centroid** for that firm. Pickle caches are named via **`embedding_cache_path(...)`** (`EMBEDDING_CACHE_TAG` includes **`turns`** for this segmentation).

**Novelty Score** = 1 − cosine_similarity(new_segment, historical_centroid)

A score near 1.0 means the segment is semantically far from anything the firm has said historically. A score near 0 means it's the usual language.

> Default `all-MiniLM-L6-v2` is a strong general sentence encoder for cosine-distance novelty. **Nomic** (`nomic-embed-text-v1.5`) is the project-spec target; when you switch, set `EMBEDDING_MODEL_NAME`, set `EMBEDDING_CACHE_TAG` to a new label (e.g. `nomic_v15`), and rebuild caches. Do **not** use FinBERT *embeddings* for this step (sentiment geometry); FinBERT is reserved for optional **H3** tone sorts later.


In [ ]:
# Load sentence encoder — explicit GPU if available
import torch
_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ENCODE_BATCH_SIZE = 256 if _DEVICE == "cuda" else 64

print(f"Loading {EMBEDDING_MODEL_NAME} on {_DEVICE}...")
ENCODER = SentenceTransformer(EMBEDDING_MODEL_NAME, device=_DEVICE)
print(f"  device={_DEVICE}  batch_size={ENCODE_BATCH_SIZE}")
if _DEVICE == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
from tqdm.auto import tqdm

# --- Novelty geometry toggle (dev tickers) ---
# Modes:
# - chunk: novelty on full chunk_text (context + exec)
# - turns: novelty on EXEC turns (exec_text), while downstream uses full chunk_text
# - full_turns: novelty on ALL raw speaker turns (Operator/Analyst/Executive turn_text)
NOVELTY_AB_COMPARE = True
NOVELTY_PRIMARY_MODE = "turns"  # "turns" | "chunk" | "full_turns"


def _novelty_cache_path_for(mode: str) -> Path:
    base = embedding_cache_path(f"novelty_scores_{mode}")
    return nh.dev_scoped_cache_path(base, DEV_TICKERS)


def _compute_novelty_records_for(mode: str) -> list[dict]:
    assert mode in ("chunk", "turns", "full_turns")
    novelty_cache = _novelty_cache_path_for(mode)
    if DEV_TICKERS:
        print(f"  Novelty pickle ({mode}): {novelty_cache.name}")

    if novelty_cache.exists():
        with open(novelty_cache, "rb") as f:
            recs = pickle.load(f)
        print(f"Loaded {len(recs):,} cached novelty records ({mode})")
        return nh.filter_novelty_records(recs, DEV_TICKERS) if nh.dev_mode_active(DEV_TICKERS) else recs

    novelty_records: list[dict] = []
    firm_historical_embeddings: dict[str, np.ndarray] = {}

    for ticker, grp in tqdm(df.groupby("ticker"), desc=f"Novelty ({mode})", unit="firm"):
        grp = grp.sort_values("quarter")
        quarters_sorted = sorted(grp["quarter_str"].unique().tolist())

        for i, q in enumerate(quarters_sorted):
            if i < 2:
                hist_rows = grp[grp["quarter_str"] == q]
                for _, row in hist_rows.iterrows():
                    tid = int(row["transcriptid"])
                    if mode == "chunk":
                        segs_score = transcript_segments(row["full_transcript_text"], tid)[:60]
                        segs_score = [s for s in segs_score if len(str(s).strip()) > 20]
                    elif mode == "turns":
                        segs_score = EXEC_TURN_TEXT_BY_TID.get(tid) or []
                        segs_score = [s for s in segs_score if len(str(s).strip()) > 20][:60]
                    elif mode == "full_turns":
                        # Score on every raw turn (Operator/Analyst/Executive)
                        segs_score = FULL_TURNS_BY_TID.get(tid) or []
                        segs_score = [s for s in segs_score if len(str(s).strip()) > 20][:200]
                    else:
                        raise ValueError(f"unknown mode={mode!r}")
                    if not segs_score:
                        continue
                    embs = ENCODER.encode(
                        segs_score,
                        batch_size=ENCODE_BATCH_SIZE,
                        normalize_embeddings=True,
                        show_progress_bar=False,
                    )
                    firm_historical_embeddings[ticker] = (
                        embs if ticker not in firm_historical_embeddings else np.vstack([firm_historical_embeddings[ticker], embs])
                    )
                continue

            if ticker not in firm_historical_embeddings:
                continue

            hist_embs = firm_historical_embeddings[ticker]
            centroid = normalize(hist_embs.mean(axis=0, keepdims=True))

            new_rows = grp[grp["quarter_str"] == q]
            for _, row in new_rows.iterrows():
                tid = int(row["transcriptid"])

                if mode == "chunk":
                    segs_score = transcript_segments(row["full_transcript_text"], tid)[:80]
                    segs_score = [s for s in segs_score if len(str(s).strip()) > 20]
                    segs_full = segs_score
                    seg_idx = list(range(len(segs_full)))

                elif mode == "turns":
                    # Joint filter to keep exec/chunk/index aligned (gate on exec_text only)
                    _all_exec = EXEC_TURN_TEXT_BY_TID.get(tid) or []
                    _all_chunk = CHUNK_TEXT_BY_EXEC_TURN_BY_TID.get(tid) or []
                    _all_idx = CHUNK_INDEX_BY_EXEC_TURN_BY_TID.get(tid) or []
                    _triples = [
                        (ex, ch, ci)
                        for ex, ch, ci in zip(_all_exec, _all_chunk, _all_idx)
                        if len(str(ex).strip()) > 20
                    ][:80]
                    if not _triples:
                        continue
                    segs_score, segs_full, seg_idx = map(list, zip(*_triples))
                    seg_idx = [int(x) for x in seg_idx]

                elif mode == "full_turns":
                    segs_score = FULL_TURNS_BY_TID.get(tid) or []
                    segs_score = [s for s in segs_score if len(str(s).strip()) > 20][:200]
                    segs_full = segs_score
                    seg_idx = list(range(len(segs_full)))

                else:
                    raise ValueError(f"unknown mode={mode!r}")

                if not segs_score:
                    continue

                new_embs = ENCODER.encode(
                    segs_score,
                    batch_size=ENCODE_BATCH_SIZE,
                    normalize_embeddings=True,
                    show_progress_bar=False,
                )

                sims = cosine_similarity(new_embs, centroid).flatten()
                novelty = 1.0 - sims

                for idx, (sent_full, nov, sim) in enumerate(zip(segs_full, novelty, sims)):
                    novelty_records.append({
                        "ticker": ticker,
                        "quarter_str": q,
                        "fiscal_year": row["fiscal_year"],
                        "transcriptid": row["transcriptid"],
                        "close_to_open_return": row.get("close_to_open_return", np.nan),
                        "sentence": sent_full,
                        "chunk_index": int(seg_idx[idx]),
                        "novelty_score": float(nov),
                        "cosine_sim": float(sim),
                        "novelty_mode": mode,
                    })

                firm_historical_embeddings[ticker] = np.vstack([hist_embs, new_embs])

    with open(novelty_cache, "wb") as f:
        pickle.dump(novelty_records, f)
    print(f"Computed {len(novelty_records):,} novelty rows ({mode}) → wrote {novelty_cache.name}")
    return nh.filter_novelty_records(novelty_records, DEV_TICKERS) if nh.dev_mode_active(DEV_TICKERS) else novelty_records


novelty_records_turns = _compute_novelty_records_for("turns")
novelty_df_turns = pd.DataFrame(novelty_records_turns)
print("turns novelty", novelty_df_turns.shape)

novelty_records_full_turns = None
novelty_df_full_turns = None
if NOVELTY_AB_COMPARE:
    novelty_records_full_turns = _compute_novelty_records_for("full_turns")
    novelty_df_full_turns = pd.DataFrame(novelty_records_full_turns)
    print("full_turns novelty", novelty_df_full_turns.shape)

novelty_df_chunk = None
if NOVELTY_AB_COMPARE:
    novelty_records_chunk = _compute_novelty_records_for("chunk")
    novelty_df_chunk = pd.DataFrame(novelty_records_chunk)
    print("chunk novelty", novelty_df_chunk.shape)

# Primary novelty dataframe for downstream pipeline
if NOVELTY_PRIMARY_MODE == "turns":
    novelty_df = novelty_df_turns
elif NOVELTY_PRIMARY_MODE == "full_turns":
    novelty_df = novelty_df_full_turns
else:
    novelty_df = novelty_df_chunk

print("primary novelty", novelty_df.shape)
print(novelty_df[["ticker","quarter_str","novelty_score"]].describe())


In [ ]:
# Distribution of novelty scores
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall distribution
axes[0].hist(novelty_df["novelty_score"], bins=60, color="#f5a623", edgecolor="none", alpha=0.85)
axes[0].set_title("Distribution of turn-level novelty scores", fontsize=12)
axes[0].set_xlabel("Novelty Score (1 − cosine similarity)")
axes[0].set_ylabel("Count")
axes[0].axvline(0.6, color="red", linestyle="--", label="Novel threshold (0.6)")
axes[0].legend()

# By firm — top 15 most novel firms on average
firm_avg = (novelty_df.groupby("ticker")["novelty_score"]
            .mean()
            .sort_values(ascending=False)
            .head(15))
axes[1].barh(firm_avg.index[::-1], firm_avg.values[::-1], color="#4ecdc4")
axes[1].set_title("Mean turn novelty score by firm (top 15)", fontsize=12)
axes[1].set_xlabel("Mean Novelty Score")

plt.tight_layout()
plt.savefig("novelty_distribution.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved novelty_distribution.png")


## 4. Salience + LLM intent filter — PDF §3.3 Stage 1 (precision gate)

A high novelty score confirms the firm is discussing something *new* — but not whether it matters financially. A novel turn about litigation, regulatory disclosure, or a CEO aside would all score as novel.

We apply the **salience dictionary** to high-novelty **segments** (turn text in the `sentence` column) to verify that the novel content is a genuine business catalyst. The dictionary scores text across 26 financial clusters (guidance, earnings performance, revenue growth, new products, partnerships, etc.) — a strong proxy for the LLM binary intent filter described in the strategy paper.

**Ollama (local LLM)** uses the **verbatim YES/NO prompt** from **`project_specification_rio_zac.pdf`** §3.3 (financial materiality vs routine call language; exclusions for boilerplate and purely retrospective results). Cache stem: **`llm_intent_cache_project_spec_turns.pkl`** (sentence-split runs used `llm_intent_cache_project_spec.pkl`).

**Signal criteria:**
- `novelty_score ≥ 0.60` — semantically distant from firm's historical baseline
- `salience_score ≥ 0.30` — meaningful financial content, not boilerplate

**Ollama:** Start or restart the local API (default `127.0.0.1:11434`):

- **Terminal:** from `final_project`, run `bash scripts/ensure_ollama.sh`.
- **Notebook:** in the **next** code cell, set `ENSURE_OLLAMA = True` and run it once (it runs the same script via `subprocess`; kills the listener on `OLLAMA_PORT` then starts `ollama serve`). Leave `False` if the menu-bar app already serves the API.

Override with env vars `OLLAMA_HOST`, `OLLAMA_PORT`, or `OLLAMA_URL` if needed. Pull your model once: `ollama pull llama3.1` (or match `OLLAMA_MODEL`).


In [ ]:
import subprocess
from pathlib import Path

# True = run scripts/ensure_ollama.sh (stops listener on OLLAMA_PORT, starts `ollama serve`).
# False = assume Ollama is already up (e.g. menu bar app on 11434).
ENSURE_OLLAMA = True

def _run_ensure_ollama_script() -> None:
    root = Path(".").resolve()
    script = root / "scripts" / "ensure_ollama.sh"
    if not script.is_file():
        print(f"Missing {script} (cwd={root})")
        return
    try:
        r = subprocess.run(
            ["bash", str(script)],
            cwd=str(root),
            capture_output=True,
            text=True,
            timeout=120,
        )
    except subprocess.TimeoutExpired:
        print("ensure_ollama.sh timed out after 120s")
        return
    if r.stdout.strip():
        print(r.stdout.rstrip())
    if r.stderr.strip():
        print(r.stderr.rstrip())
    if r.returncode != 0:
        print(f"ensure_ollama.sh exit code {r.returncode}")
    else:
        print("ensure_ollama.sh completed.")

if ENSURE_OLLAMA:
    _run_ensure_ollama_script()

In [ ]:
import os
import time
import requests, json
from tqdm.auto import tqdm

# Default matches `ollama serve` (run: bash scripts/ensure_ollama.sh from final_project/)
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "127.0.0.1")
OLLAMA_PORT = os.environ.get("OLLAMA_PORT", "11434")
OLLAMA_URL = os.environ.get(
    "OLLAMA_URL",
    f"http://{OLLAMA_HOST}:{OLLAMA_PORT}/api/generate",
)
OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "llama3.1")

print(f"Ollama API: {OLLAMA_URL} | model: {OLLAMA_MODEL}")

# Overnight-friendly: retries, long read timeout, periodic cache checkpoints (resume by re-running this cell).
OLLAMA_HTTP_TIMEOUT = float(os.environ.get("OLLAMA_HTTP_TIMEOUT", "120"))  # full request; local gen can be slow
OLLAMA_MAX_RETRIES = int(os.environ.get("OLLAMA_MAX_RETRIES", "8"))
OLLAMA_RETRY_BASE_SEC = float(os.environ.get("OLLAMA_RETRY_BASE_SEC", "2.0"))
OLLAMA_RETRY_MAX_SLEEP = float(os.environ.get("OLLAMA_RETRY_MAX_SLEEP", "120"))
LLM_CACHE_SAVE_EVERY = int(os.environ.get("LLM_CACHE_SAVE_EVERY", "25"))  # disk write after this many *new* API answers
_ollama_session = requests.Session()

# Thresholds match `project_specification_rio_zac.pdf` §3.3 Stage 1 (precision gate).
NOVELTY_THRESHOLD  = 0.60
SALIENCE_THRESHOLD = 0.30

# Cache stem tied to verbatim PDF prompt; bump filename if §3.3 prompt changes.
LLM_INTENT_CACHE = nh.dev_scoped_cache_path(Path("./pkl_cache/llm_intent_cache_project_spec_turns.pkl"), DEV_TICKERS)
if DEV_TICKERS:
    print(f"  LLM intent cache: {LLM_INTENT_CACHE.name}")

def _parse_llm_yes_no(raw: str) -> bool:
    """Parse YES/NO from model output (PDF asks for one word only)."""
    t = (raw or "").strip().upper()
    if not t:
        return False
    first_line = t.split("\n", 1)[0].strip()
    for tok in first_line.replace(",", " ").split():
        if tok.startswith("YES"):
            return True
        if tok.startswith("NO"):
            return False
    return False


def ollama_intent_check(sentence: str) -> bool:
    """Verbatim prompt from `project_specification_rio_zac.pdf` §3.3; retries transient HTTP failures."""
    prompt = (
        f'You are a financial analyst reviewing a sentence from an earnings call transcript. Sentence: "{sentence}"\n\n'
        "Does this sentence contain information that is likely to be financially material and meaningfully different from "
        "routine earnings call language? This includes, but is not limited to, new strategic initiatives, product or service "
        "changes, partnerships, contracts, market expansion, pricing decisions, cost actions, management changes, "
        "capital allocation shifts, or any forward-looking statement that represents a departure from prior guidance or "
        "company narrative. Exclude only clear boilerplate (safe-harbour disclaimers, operator instructions, generic "
        "pleasantries) and retrospective descriptions of already-reported results with no forward-looking content. Reply "
        "with ONLY one word: YES or NO."
    )
    payload = {
        "model"  : OLLAMA_MODEL,
        "prompt" : prompt,
        "stream" : False,
        "options": {"temperature": 0.0, "num_predict": 32},
    }
    last_err = None
    for attempt in range(OLLAMA_MAX_RETRIES):
        try:
            resp = _ollama_session.post(OLLAMA_URL, json=payload, timeout=OLLAMA_HTTP_TIMEOUT)
            if resp.status_code >= 500:
                raise requests.HTTPError(f"HTTP {resp.status_code}")
            return _parse_llm_yes_no(resp.json().get("response", ""))
        except (requests.Timeout, requests.ConnectionError, requests.exceptions.ChunkedEncodingError) as e:
            last_err = e
        except requests.HTTPError as e:
            last_err = e
        except ValueError as e:
            last_err = e
        except Exception as e:
            print(f"Ollama non-retry error: {e}")
            return False
        wait = min(OLLAMA_RETRY_MAX_SLEEP, OLLAMA_RETRY_BASE_SEC * (2 ** attempt))
        print(f"Ollama retry {attempt + 1}/{OLLAMA_MAX_RETRIES} ({last_err!r}); sleep {wait:.1f}s")
        time.sleep(wait)
    print(f"Ollama failed after {OLLAMA_MAX_RETRIES} attempts: {last_err}")
    return False

# ── Load LLM cache ─────────────────────────────────────────────────────────────
if LLM_INTENT_CACHE.exists():
    with open(LLM_INTENT_CACHE, "rb") as f:
        intent_cache = pickle.load(f)
    print(f"Loaded {len(intent_cache):,} cached LLM results")
else:
    intent_cache = {}

# ── Step 1: salience filter on high-novelty segments ───────────────────────────
# We cache salience by (sentence, fiscal_year) so A/B novelty modes don't recompute.

_SAL_CACHE: dict[tuple[str, int], tuple[float, str | None, str]] = {}


def _score_salience_rows(df_in: pd.DataFrame) -> pd.DataFrame:
    if df_in.empty:
        return df_in.copy()
    out = []
    for _, row in tqdm(df_in.iterrows(), total=len(df_in), desc="Salience scoring", leave=False):
        sent = row["sentence"]
        yr = int(row["fiscal_year"]) if pd.notna(row.get("fiscal_year")) else 2020
        key = (str(sent), int(yr))
        if key in _SAL_CACHE:
            score, top_cluster, breakdown_json = _SAL_CACHE[key]
        else:
            score, breakdown = score_sentence(str(sent), year=yr)
            top_cluster = max(breakdown, key=breakdown.get) if breakdown else None
            breakdown_json = json.dumps(breakdown)
            _SAL_CACHE[key] = (float(score), top_cluster, breakdown_json)
        out.append({
            "salience_score": float(score),
            "top_cluster": top_cluster,
            "cluster_breakdown": breakdown_json,
        })
    sal_df = pd.DataFrame(out)
    return pd.concat([df_in.reset_index(drop=True), sal_df.reset_index(drop=True)], axis=1)


def _build_signal_df(novelty_df_in: pd.DataFrame, *, label: str) -> pd.DataFrame:
    high = novelty_df_in[novelty_df_in["novelty_score"] >= NOVELTY_THRESHOLD].copy()
    print(f"[{label}] High-novelty segments: {len(high):,}")
    if high.empty:
        return high
    print(f"[{label}] Scoring salience...")
    scored = _score_salience_rows(high)
    scored = scored[scored["salience_score"] >= SALIENCE_THRESHOLD].copy()
    print(f"[{label}] After salience filter: {len(scored):,} segments")
    return scored


# Compare Top-10 candidates under both novelty geometries.
signal_df_turns = _build_signal_df(novelty_df_turns, label="turns")

signal_df_full_turns = None
if 'novelty_df_full_turns' in globals() and novelty_df_full_turns is not None:
    signal_df_full_turns = _build_signal_df(novelty_df_full_turns, label="full_turns")

signal_df_chunk = None
if 'novelty_df_chunk' in globals() and novelty_df_chunk is not None:
    signal_df_chunk = _build_signal_df(novelty_df_chunk, label="chunk")

def _print_top10(df_in: pd.DataFrame, *, label: str) -> None:
    if df_in is None or df_in.empty:
        print(f"\n=== Top 10 ({label}) — EMPTY ===\n")
        return
    top = df_in.sort_values(["novelty_score", "salience_score"], ascending=False).head(10)
    print(f"\n=== Top 10 Most Novel + Salient Segments ({label}) ===\n")
    for _, r in top.iterrows():
        print(f"Ticker : {r['ticker']} | Quarter: {r['quarter_str']} | Transcript: {int(r['transcriptid'])} | Chunk: {int(r['chunk_index'])}")
        print(f"Novelty: {r['novelty_score']:.3f}  |  Salience: {r['salience_score']:.3f}  |  Cluster: {r['top_cluster']}")
        print(f"Text : {str(r['sentence'])[:4000]}")
        print("")

_print_top10(signal_df_turns, label="turns")
if signal_df_full_turns is not None:
    _print_top10(signal_df_full_turns, label="full_turns")
if signal_df_chunk is not None:
    _print_top10(signal_df_chunk, label="chunk")

# Primary downstream pipeline uses the chosen novelty mode.
if NOVELTY_PRIMARY_MODE == "turns":
    signal_df = signal_df_turns
elif NOVELTY_PRIMARY_MODE == "full_turns":
    signal_df = signal_df_full_turns
else:
    signal_df = signal_df_chunk

# Merge rolling analyst attention onto each (call, cluster) with no lookahead.
# `analyst_roll` is built earlier from raw_turn_df (Analysts only) over trailing 90 days.
if 'analyst_roll' in globals() and len(analyst_roll):
    signal_df = signal_df.merge(
        analyst_roll,
        on=['transcriptid','top_cluster'],
        how='left',
        validate='many_to_one',
    )
else:
    signal_df['trailing_unique_analysts_90d'] = np.nan
signal_df['trailing_unique_analysts_90d'] = signal_df['trailing_unique_analysts_90d'].fillna(0).astype(int)
print(f"After salience filter: {len(signal_df):,} segments")

# ── Step 2: LLM intent filter (PDF §3.3 Stage 1 — all rows that passed novelty≥0.60 & salience≥0.30)
llm_candidates = signal_df.copy()

print(f"Sending {len(llm_candidates):,} segments to Ollama (spec precision gate)...")
print(
    f"  timeout={OLLAMA_HTTP_TIMEOUT}s retries={OLLAMA_MAX_RETRIES} save_every={LLM_CACHE_SAVE_EVERY} new answers"
)

llm_pass = []
_new_since_save = 0
_total = len(llm_candidates)
_pbar = tqdm(total=_total, desc="LLM intent filter", unit="seg")
try:
    for _i, (_, row) in enumerate(llm_candidates.iterrows(), start=1):
        sent = row["sentence"]
        if sent not in intent_cache:
            intent_cache[sent] = ollama_intent_check(sent)
            _new_since_save += 1
            if _new_since_save >= LLM_CACHE_SAVE_EVERY:
                with open(LLM_INTENT_CACHE, "wb") as f:
                    pickle.dump(intent_cache, f)
                print(f"  checkpoint: {len(intent_cache):,} cached keys → {LLM_INTENT_CACHE.name}")
                _new_since_save = 0
        llm_pass.append(intent_cache[sent])
        _pbar.update(1)
finally:
    _pbar.close()
    with open(LLM_INTENT_CACHE, "wb") as f:
        pickle.dump(intent_cache, f)
    print(f"Cache saved: {len(intent_cache):,} keys → {LLM_INTENT_CACHE.name} (safe to stop kernel; re-run cell to resume)")

llm_candidates["llm_intent_pass"] = llm_pass

signal_df = llm_candidates[llm_candidates["llm_intent_pass"] == True].copy()

print(f"After LLM intent filter: {len(signal_df):,} segments")
print(f"LLM pass rate: {llm_candidates['llm_intent_pass'].mean():.1%}")


In [ ]:
# What clusters dominate novel sentences?
cluster_counts = (signal_df["top_cluster"]
                  .value_counts()
                  .head(15))

plt.figure(figsize=(10, 5))
plt.barh(cluster_counts.index[::-1], cluster_counts.values[::-1], color="#a78bfa")
plt.title("Top Financial Clusters in Novel + Salient Sentences", fontsize=12)
plt.xlabel("Sentence count")
plt.tight_layout()
plt.savefig("cluster_distribution.png", dpi=120, bbox_inches="tight")
plt.show()

print("\nTop 10 clusters in novel sentences:")
print(cluster_counts.head(10).to_string())


In [ ]:
# Example: most novel + salient segments
print("=== Top 10 Most Novel + Salient Segments ===\n")
top_examples = (signal_df
                .sort_values(["novelty_score", "salience_score"], ascending=False)
                .head(10))

for _, row in top_examples.iterrows():
    tid = int(row['transcriptid'])
    ci = int(row.get('chunk_index', -1)) if pd.notna(row.get('chunk_index', np.nan)) else -1
    print(f"Ticker : {row['ticker']}  |  Quarter: {row['quarter_str']} | Transcript: {tid} | Chunk: {ci}")
    print(f"Novelty: {row['novelty_score']:.3f}  |  Salience: {row['salience_score']:.3f}  |  Cluster: {row['top_cluster']}")

    # Show chunk parts (context + exec) when available; else fall back to the stored segment text.
    ctx = ''
    ex = ''
    if ci >= 0:
        ctx_list = CONTEXT_TEXT_BY_TID.get(tid)
        ex_list = EXEC_TEXT_BY_TID.get(tid)
        if isinstance(ctx_list, list) and ci < len(ctx_list):
            ctx = str(ctx_list[ci])
        if isinstance(ex_list, list) and ci < len(ex_list):
            ex = str(ex_list[ci])

    if ctx or ex:
        if ctx:
            print(f"Context: {ctx[:400]}...")
        if ex:
            print(f"Exec   : {ex[:400]}...")
    else:
        print(f"Text   : {str(row['sentence'])[:800]}...")
    print()


## 5. Cross-market frequency penalty — PDF §3.3 (§4.4 look-ahead note)

When many firms in the same quarter share the same dominant novelty cluster, the topic is more **systemic** than **idiosyncratic**. The memo specifies a **smooth sigmoid** penalty in cross-market frequency (not a hard cliff at 40%); see **`project_specification_rio_zac.pdf`** §3.3. The notebook still uses **quarter × cluster** counts as a practical proxy until a trailing three-month prior window is merged from timestamps.

The penalty is applied at the **cluster × quarter** level: for each quarter, compute the fraction of firms where a given cluster is the dominant novelty cluster. If that fraction exceeds the threshold, reduce the signal weight proportionally.


In [ ]:
# PDF §3.3 sigmoid: penalty_mult(f) = 1 / (1 + exp(k * (f - f0)))
MACRO_PENALTY_F0 = 0.40
MACRO_PENALTY_K = 15.0

_n_sig = len(signal_df)
signal_df, cross_section = nh.macro_adjust_signal_pipeline(
    signal_df, penalty_f0=MACRO_PENALTY_F0, penalty_k=MACRO_PENALTY_K
)
print(f"Dropped {_n_sig - len(signal_df):,} rows with missing top_cluster before penalty merge")

print("Systemic topics by quarter (cross-market frequency ≥ threshold):")
systemic = cross_section[cross_section["is_systemic"]].sort_values(
    "cross_market_freq", ascending=False
)
print(systemic[["quarter_str", "top_cluster", "n_firms", "total_firms",
                "cross_market_freq"]].head(20).to_string(index=False))

print("\nSignal score distribution after macro adjustment:")
print(signal_df["adjusted_novelty"].describe().round(4))


In [ ]:
# Penalty merge + adjusted_novelty: done in nh.macro_adjust_signal_pipeline (cell above).


## 6. Firm-quarter signal & adjusted novelty — PDF §3.3 Stage 2 + aggregation

Aggregate to the **firm-quarter** level. The signal is:

```
adjusted_novelty_score(firm, quarter) =
    mean(novelty × salience × macro_penalty) over all novel+salient sentences
```

We merge **`close_to_open_return`** for the overnight reaction, then attach **`permno`**, **`gvkey`**, **`word_count`**, **`sue`** (I/B/E/S), **`pre_drift`**, and surprise fields from the cleaned **`df`** row for that event. **CAR windows** use winsorized dailies from §1b:

- **`CAR_m1_p1`** = `ret_t-1` + `ret_t0` + `ret_t1` (event window [−1, +1])
- **`CAR_p2_p5`** = `ret_t2` + `ret_t3` + `ret_t4` + `ret_t5` (post window [+2, +5])

Daily `ret_t*` ingredients are dropped after summing to keep the firm-quarter table narrow; re-merge from `df` if you need them in regressions.


In [ ]:
# Aggregate to firm-quarter signal
assert "penalty_mult" in signal_df.columns, (
    "penalty_mult missing — run the macro-adjustment merge cell above first."
)


firm_quarter_signal = (signal_df
    .groupby(["ticker", "quarter_str", "fiscal_year"])
    .agg(
        mean_novelty          = ("novelty_score",   "mean"),
        mean_salience         = ("salience_score",  "mean"),
        mean_adjusted_novelty = ("adjusted_novelty","mean"),
        n_novel_turns         = ("sentence",        "count"),
        macro_penalty         = ("penalty_mult",    "mean"),
        dominant_cluster      = ("top_cluster",     nh.first_mode),
        close_to_open_return  = ("close_to_open_return", "first"),
    )
    .reset_index()
)

firm_quarter_signal = nh.merge_event_panel_into_firm_quarter(firm_quarter_signal, df)

print(f"Firm-quarter signals: {len(firm_quarter_signal):,}")
print(f"  Panel merge — permno non-null: {firm_quarter_signal['permno'].notna().sum():,}")
print(f"  CAR[-1,+1]  (CAR_m1_p1) non-null: {firm_quarter_signal['CAR_m1_p1'].notna().sum():,}")
print(f"  CAR[+2,+5]  (CAR_p2_p5) non-null: {firm_quarter_signal['CAR_p2_p5'].notna().sum():,}")
print(f"  sue non-null: {firm_quarter_signal['sue'].notna().sum():,}")
print(f"  pre_drift non-null: {firm_quarter_signal['pre_drift'].notna().sum():,}")

print(firm_quarter_signal[
    ["ticker", "quarter_str", "mean_adjusted_novelty", "close_to_open_return",
     "CAR_m1_p1", "CAR_p2_p5", "sue", "pre_drift"]
].head(10).to_string(index=False))


In [ ]:
# Rank into quintiles each quarter (cross-sectional signal)
firm_quarter_signal["novelty_quintile"] = (
    firm_quarter_signal
    .groupby("quarter_str")["mean_adjusted_novelty"]
    .transform(nh.quintile_within_quarter)
)

# Clean returns (legacy ±15% clip on overnight — align with §1b winsorization when you standardize)
fqs = firm_quarter_signal.dropna(subset=["close_to_open_return", "novelty_quintile"]).copy()
fqs["close_to_open_return"] = fqs["close_to_open_return"].clip(-0.15, 0.15)
fqs["novelty_quintile"] = fqs["novelty_quintile"].astype(float)

print(f"Observations with returns + quintile: {len(fqs):,}")
print(fqs.groupby("novelty_quintile")["close_to_open_return"].describe())

# Subset with full CAR + SUE for event-study / FE regressions (optional)
fqs_reg = fqs.dropna(subset=["CAR_m1_p1", "CAR_p2_p5", "sue"]).copy()
print(f"\nWith CAR[-1,+1], CAR[+2,+5], and SUE (for regressions): {len(fqs_reg):,}")


## 6b. Model specification — PDF §7 (regression hooks via `novelty_helpers`)

- **`nh.fit_ols_cluster(formula, data, cluster_col)`** — OLS with clustered covariance (here `cluster_col="quarter_str"` as a coarse calendar proxy; swap to event date when merged).
- **`nh.fetch_ff5_monthly`** — monthly Fama–French 5 + RF (Ken French zip; works without `pandas_datareader`).
- **`nh.ff5_alpha_regression(ls_monthly, factors=...)`** — time-series alpha on a **monthly** long–short return (decimals). Replace the demo series with portfolio returns from your calendar-time sort.

Requires **`statsmodels`** (`pip install statsmodels`).

In [ ]:
# --- Clustered OLS on firm-quarter panel ---
_reg = fqs_reg.dropna(
    subset=["CAR_m1_p1", "mean_adjusted_novelty", "sue", "pre_drift", "word_count"]
).copy()
_reg["log_wc"] = np.log(_reg["word_count"].clip(lower=1))

try:
    if len(_reg) > 50:
        fit_car = nh.fit_ols_cluster(
            "CAR_m1_p1 ~ mean_adjusted_novelty + sue + pre_drift + log_wc",
            data=_reg,
            cluster_col="quarter_str",
        )
        print(fit_car.summary())
    else:
        print(f"Skip clustered OLS: only {len(_reg)} complete rows in fqs_reg")
except Exception as ex:
    print("Clustered OLS skipped:", ex)

# --- FF5 template: replace demo_ls with monthly LS portfolio returns ---
try:
    fac = nh.fetch_ff5_monthly(start="2018-01-01", end="2024-12-31")
    demo_ls = pd.Series(0.005, index=fac.index)
    ff_fit = nh.ff5_alpha_regression(demo_ls, factors=fac, excess_returns=False)
    print("\nDemo FF5 monthly intercept (decimal):", float(ff_fit.params["const"]))
except Exception as ex:
    print("FF5 demo skipped:", ex)

## 7. Returns around the event — PDF §6.4 (CAR / event supplement; §6.1–6.3 portfolio extension)

The strategy thesis: **high-novelty firms experience a market overreaction** — an initial spike that mean-reverts. We test this by examining close-to-open returns (overnight market reaction) across novelty quintiles.

**Charts:** (1) mean *signed* overnight return by quintile; (2) mean **absolute** overnight return by quintile (typical magnitude of the move, ignoring direction); (3) cumulative long Q1 / short Q5.

The short signal targets **Quintile 5** (highest novelty) — firms discussing something genuinely new that the market overprices on the announcement.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Mean signed return by quintile ─────────────────────────────────────────
quintile_returns = (fqs.groupby("novelty_quintile")["close_to_open_return"]
                    .agg(["mean","sem","count"])
                    .reset_index())

colors = ["#34d399","#60a5fa","#f5a623","#fb923c","#f87171"]
bars = axes[0].bar(
    quintile_returns["novelty_quintile"],
    quintile_returns["mean"] * 100,
    color=colors,
    alpha=0.85, edgecolor="none"
)
axes[0].errorbar(
    quintile_returns["novelty_quintile"],
    quintile_returns["mean"] * 100,
    yerr=quintile_returns["sem"] * 100 * 1.96,
    fmt="none", color="white", capsize=4, linewidth=1.5
)
axes[0].axhline(0, color="white", linewidth=0.8, linestyle="--", alpha=0.4)
axes[0].set_xlabel("Novelty Quintile (1=low, 5=high)")
axes[0].set_ylabel("Mean Close-to-Open Return (%)")
axes[0].set_title("Mean Overnight Return by Novelty Quintile")
axes[0].set_xticks([1,2,3,4,5])
for bar, (_, row) in zip(bars, quintile_returns.iterrows()):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.02,
                 f"n={int(row['count'])}",
                 ha="center", va="bottom", fontsize=8)

# ── Mean absolute return by quintile (magnitude of overnight move) ───────
quintile_abs = (
    fqs.groupby("novelty_quintile")["close_to_open_return"]
    .agg(
        mean_abs=lambda s: float(s.abs().mean()),
        sem_abs=lambda s: float(s.abs().sem()) if len(s) > 1 else 0.0,
        count="count",
    )
    .reset_index()
)
# Fix sem for groups with n=1 (sem is NaN)
quintile_abs["sem_abs"] = quintile_abs["sem_abs"].fillna(0.0)

bars_abs = axes[1].bar(
    quintile_abs["novelty_quintile"],
    quintile_abs["mean_abs"] * 100,
    color=colors,
    alpha=0.85,
    edgecolor="none",
)
axes[1].errorbar(
    quintile_abs["novelty_quintile"],
    quintile_abs["mean_abs"] * 100,
    yerr=quintile_abs["sem_abs"] * 100 * 1.96,
    fmt="none",
    color="white",
    capsize=4,
    linewidth=1.5,
)
axes[1].set_xlabel("Novelty Quintile (1=low, 5=high)")
axes[1].set_ylabel("Mean |Close-to-Open Return| (%)")
axes[1].set_title("Mean Absolute Overnight Return by Quintile")
axes[1].set_xticks([1, 2, 3, 4, 5])
for bar, (_, row) in zip(bars_abs, quintile_abs.iterrows()):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"n={int(row['count'])}",
        ha="center",
        va="bottom",
        fontsize=8,
    )

# ── Long-short backtest: Q1 long, Q5 short ────────────────────────────────
fqs_sorted = fqs.sort_values("quarter_str")
ls_by_quarter = (fqs_sorted[fqs_sorted["novelty_quintile"].isin([1,5])]
    .groupby(["quarter_str","novelty_quintile"])["close_to_open_return"]
    .mean()
    .unstack()
    .rename(columns={1:"long_Q1", 5:"short_Q5"})
    .dropna()
)
ls_by_quarter["long_short"] = ls_by_quarter["long_Q1"] - ls_by_quarter["short_Q5"]
ls_by_quarter["cumulative_ls"] = (1 + ls_by_quarter["long_short"]).cumprod() - 1

axes[2].plot(range(len(ls_by_quarter)),
             ls_by_quarter["cumulative_ls"] * 100,
             color="#f5a623", linewidth=2.5)
axes[2].fill_between(range(len(ls_by_quarter)),
                      0, ls_by_quarter["cumulative_ls"] * 100,
                      alpha=0.2, color="#f5a623")
axes[2].axhline(0, color="white", linewidth=0.8, linestyle="--", alpha=0.4)
axes[2].set_xlabel("Quarter (chronological)")
axes[2].set_ylabel("Cumulative Return (%)")
axes[2].set_title("Long Q1 / Short Q5 Cumulative Return")
axes[2].set_xticks(range(0, len(ls_by_quarter), max(1, len(ls_by_quarter)//8)))
axes[2].set_xticklabels(
    ls_by_quarter.index[::max(1, len(ls_by_quarter)//8)],
    rotation=45, ha="right", fontsize=8
)

plt.tight_layout()
plt.savefig("backtest_results.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Summary statistics
print("=" * 55)
print("STRATEGY PERFORMANCE SUMMARY")
print("=" * 55)

q1 = fqs[fqs["novelty_quintile"] == 1]["close_to_open_return"]
q5 = fqs[fqs["novelty_quintile"] == 5]["close_to_open_return"]
ls = ls_by_quarter["long_short"]

def sharpe(returns, periods_per_year=4):
    """Annualized Sharpe (quarterly observations)."""
    return (returns.mean() / returns.std()) * (periods_per_year ** 0.5)

print(f"{'Metric':<35} {'Q1 (Low Nov.)':>14} {'Q5 (High Nov.)':>15} {'L-S':>10}")
print("-" * 75)
print(f"{'Mean return':<35} {q1.mean()*100:>13.3f}% {q5.mean()*100:>14.3f}% "
      f"{ls.mean()*100:>9.3f}%")
print(f"{'Std dev':<35} {q1.std()*100:>13.3f}% {q5.std()*100:>14.3f}% "
      f"{ls.std()*100:>9.3f}%")
print(f"{'Hit rate (return > 0)':<35} {(q1>0).mean():>14.1%} {(q5>0).mean():>14.1%} "
      f"{(ls>0).mean():>9.1%}")
print(f"{'Annualized Sharpe':<35} {sharpe(q1):>14.3f} {sharpe(q5):>14.3f} "
      f"{sharpe(ls):>9.3f}")
print(f"{'Observations':<35} {len(q1):>14,} {len(q5):>14,} {len(ls):>9,}")
print("=" * 55)


## 8. Diagnostics — PDF §5 (EDA-style)

### 8.1 Novelty by Cluster — which clusters drive the short signal?


In [ ]:
# Q5 dominant clusters
q5_firms = fqs[fqs["novelty_quintile"] == 5]
q5_clusters = (q5_firms["dominant_cluster"]
               .value_counts()
               .head(12))

plt.figure(figsize=(10, 4))
plt.barh(q5_clusters.index[::-1], q5_clusters.values[::-1], color="#f87171", alpha=0.85)
plt.title("Dominant Salience Clusters in Q5 (Highest Novelty) Firms", fontsize=12)
plt.xlabel("Firm-quarter count")
plt.tight_layout()
plt.savefig("q5_clusters.png", dpi=120, bbox_inches="tight")
plt.show()

print("Interpretation: clusters dominating Q5 represent the types of topics")
print("that markets tend to overprice when firms discuss them for the first time.")


### 8.2 Novelty score vs. return scatter

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(
    fqs["mean_adjusted_novelty"],
    fqs["close_to_open_return"] * 100,
    alpha=0.3, s=15, color="#a78bfa"
)

# OLS trend line
from numpy.polynomial import polynomial as P
x = fqs["mean_adjusted_novelty"].values
y = fqs["close_to_open_return"].values * 100
mask = np.isfinite(x) & np.isfinite(y)
coef = np.polyfit(x[mask], y[mask], 1)
xline = np.linspace(x[mask].min(), x[mask].max(), 100)
plt.plot(xline, np.polyval(coef, xline), color="#f5a623", linewidth=2,
         label=f"OLS slope: {coef[0]:.3f}")
plt.axhline(0, color="white", linewidth=0.7, linestyle="--", alpha=0.4)
plt.xlabel("Adjusted Novelty Score (novelty × salience × macro_penalty)")
plt.ylabel("Close-to-Open Return (%)")
plt.title("Novelty Score vs. Overnight Market Reaction")
plt.legend()
plt.tight_layout()
plt.savefig("novelty_vs_return.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"OLS coefficient: {coef[0]:.4f}")
print("Negative slope supports thesis: higher novelty → more overreaction → lower next-day open")


## 9. Example signals (illustration)

Pull the top short candidates from the most recent quarter — firms with the highest adjusted novelty scores that passed both the salience and macro filters.


In [ ]:
most_recent_q = fqs["quarter_str"].max()
top_shorts = (
    fqs[fqs["quarter_str"] == most_recent_q]
    .nlargest(10, "mean_adjusted_novelty")
    [["ticker","quarter_str","mean_adjusted_novelty","mean_novelty",
      "mean_salience","macro_penalty","dominant_cluster",
      "n_novel_turns","close_to_open_return"]]
)

print(f"Top Short Candidates — {most_recent_q}")
print(top_shorts.to_string(index=False))


In [ ]:
# Show the specific novel sentences for the top signal
top_ticker = top_shorts.iloc[0]["ticker"]
top_q      = top_shorts.iloc[0]["quarter_str"]

top_sents = (signal_df[
    (signal_df["ticker"] == top_ticker) &
    (signal_df["quarter_str"] == top_q)
]
.nlargest(5, "adjusted_novelty")
[["sentence","novelty_score","salience_score","top_cluster","adjusted_novelty"]])

print(f"\n=== Top novel sentences: {top_ticker} {top_q} ===\n")
for _, row in top_sents.iterrows():
    print(f"Cluster  : {row['top_cluster']}")
    print(f"Novelty  : {row['novelty_score']:.3f}  |  Salience: {row['salience_score']:.3f}  "
          f"|  Adjusted: {row['adjusted_novelty']:.4f}")
    print(f"Sentence : {row['sentence'][:200]}")
    print()


## 10. Discussion — PDF §2 mechanism & implementation gaps

### What this implements from the strategy paper
| Step | Paper | Implementation |
|------|-------|----------------|
| Baseline Build | Per-firm TF-IDF vocabulary profile | ✅ Rolling TF-IDF per firm-quarter |
| Semantic Check | Sentence embeddings + cosine distance | ✅ `EMBEDDING_MODEL_NAME` (default MiniLM; swap to Nomic via config + new cache tag) |
| Text unit | Turn-level (spec) | ⚠️ **Sentence splits** on `full_transcript_text` — `FINAL.csv` has no component/turn rows |
| Empirical panel | `FINAL.csv` + controls | ✅ Returns window + I/B/E/S (`ibes_sue_eps`, surprises); **market cap** still needs merge |
| Intent Verify | LLM binary classifier (YES/NO) | ✅ Salience dictionary (26 clusters, weighted) |
| Macro Adjust | Cross-sectional frequency discount | ✅ Cluster × quarter firm-frequency penalty |

### Why `all-MiniLM-L6-v2` and not FinBERT for embeddings
FinBERT is fine-tuned on financial text for **sentiment classification** (positive/negative/neutral). Its internal representations encode sentiment polarity — the cosine distance between two FinBERT embeddings measures how different their *sentiment* is, not how *semantically novel* one sentence is relative to historical context. Using FinBERT embeddings for novelty detection would flag positive-vs-negative sentence pairs as "novel" even if they discuss identical topics, and would fail to flag genuinely new vocabulary.

`all-MiniLM-L6-v2` is trained on paraphrase and semantic textual similarity tasks — its embedding space preserves meaning such that cosine distance is a valid measure of semantic novelty.

### Limitations & extensions
- **Look-ahead bias:** Ensure baseline only includes transcripts strictly before the event quarter (✅ implemented)
- **Model staleness:** A firm that legitimately pivots will generate persistent false signals — re-weight baseline with recency decay
- **LLM intent filter:** The salience dictionary is a strong proxy but cannot perform the nuanced binary YES/NO classification the paper envisions — a Claude/GPT call on the top-5 novel sentences per firm would add precision
- **Mean reversion horizon T:** This analysis uses the overnight close-to-open return; the paper implies holding for T days — extending to 5/10/20-day cumulative returns is a natural next step
- **Short execution:** Entry timing (after spike confirmation), borrowing costs, and bid-ask spread are not modeled here


## 11. Event Study — PDF §6.4 & §7

Formal event-study analysis per the research memo:
1. **Summary statistics** — Panel A (financial) and Panel B (text), key diagnostics
2. **CAR path** by novelty quintile in [-15, +15] trading days
3. **Four regression models** (M1--M4) testing overreaction, mean reversion, and pre-priced novelty
4. **Fama--French 5-factor alpha** for the calendar-time long-short portfolio

CARs are pre-computed from `FINAL.csv` daily returns (`ret_t-15` ... `ret_t+15`) via `novelty_helpers.build_event_panel_extras()`. Returns winsorized at 1st/99th percentile in the sample-construction step (cell 7). Standard errors clustered by calendar quarter.

In [ ]:
# ── Build regression-ready dataframe with all M1-M4 variables ────────
import statsmodels.formula.api as smf

reg = fqs_reg.dropna(subset=[
    "CAR_m1_p1", "CAR_p2_p5", "mean_adjusted_novelty",
    "sue", "pre_drift", "word_count",
]).copy()

# Derived regressors
reg["log_wc"]          = np.log(reg["word_count"].clip(lower=1))
reg["nov_score"]       = reg["mean_adjusted_novelty"]
reg["nov_score_sq"]    = reg["nov_score"] ** 2
reg["abs_pre_drift"]   = reg["pre_drift"].abs()

# Quintile dummies (Q1 = omitted reference group)
for q in [2, 3, 4, 5]:
    reg[f"Q{q}"] = (reg["novelty_quintile"] == q).astype(int)

# Interaction terms
reg["Q5_x_car"]       = reg["Q5"] * reg["CAR_m1_p1"]
reg["nov_x_abs_pre"]  = reg["nov_score"] * reg["abs_pre_drift"]

# Fixed effects
reg["qtr_fe"] = reg["quarter_str"].str[-2:]   # Q1..Q4
reg["yr_fe"]  = reg["fiscal_year"].astype(str)

print(f"Regression sample: {len(reg):,} firm-quarter observations")
print(f"Cluster quarters: {reg['quarter_str'].nunique():,}")
print(f"\nKey variable coverage:")
for c in ["CAR_m1_p1","CAR_p2_p5","sue","pre_drift","nov_score","close_to_open_return"]:
    print(f"  {c:30s}  non-null: {reg[c].notna().sum():,}")

### 11.1 Summary Statistics — PDF §5

Panel A (financial variables) and Panel B (text variables), replicating the
EDA slide from the research memo. Plus key diagnostic ratios.

In [ ]:
# ── Panel A: Financial Variables ─────────────────────────────────────
panel_a_vars = {
    "CAR_m1_p1":            "CAR[-1,+1]",
    "CAR_p2_p5":            "CAR[+2,+5]",
    "close_to_open_return": "Close-to-Open",
    "pre_drift":            "Pre-drift [-15,-2]",
    "sue":                  "SUE (I/B/E/S)",
}

rows_a = []
for col, label in panel_a_vars.items():
    s = reg[col].dropna()
    rows_a.append({
        "Variable": label, "N": f"{len(s):,}",
        "Mean": f"{s.mean():.4f}", "Std": f"{s.std():.4f}",
        "p1":  f"{s.quantile(.01):.3f}", "p25": f"{s.quantile(.25):.3f}",
        "p50": f"{s.quantile(.50):.3f}", "p75": f"{s.quantile(.75):.3f}",
        "p99": f"{s.quantile(.99):.3f}",
    })

print("Panel A  \u00b7  Financial Variables")
print(pd.DataFrame(rows_a).to_string(index=False))

# ── Panel B: Text Variables ──────────────────────────────────────────
panel_b_vars = {
    "word_count":            "Word Count",
    "mean_adjusted_novelty": "Adjusted Novelty",
    "mean_novelty":          "Raw Novelty",
    "mean_salience":         "Salience Score",
    "n_novel_turns":         "Novel Turns",
}

rows_b = []
for col, label in panel_b_vars.items():
    s = reg[col].dropna()
    rows_b.append({
        "Variable": label, "N": f"{len(s):,}",
        "Mean": f"{s.mean():.4f}", "Std": f"{s.std():.4f}",
        "p25": f"{s.quantile(.25):.3f}", "p50": f"{s.quantile(.50):.3f}",
        "p99": f"{s.quantile(.99):.3f}",
    })

print("\nPanel B  \u00b7  Text Variables")
print(pd.DataFrame(rows_b).to_string(index=False))

# ── Key diagnostics (memo slide) ─────────────────────────────────────
# Event-day return = ret_t0 (single-day)
if "ret_t0" not in reg.columns:
    # Merge back from df if needed
    _t0 = df[["ticker","quarter_str","ret_t0"]].drop_duplicates(
        subset=["ticker","quarter_str"], keep="last")
    reg = reg.merge(_t0, on=["ticker","quarter_str"], how="left")

vol_event = reg["ret_t0"].std()
# Approximate non-event vol from the full panel's pre-event returns
_pre_cols = [f"ret_t{j}" for j in range(-15, -1)]
_pre_avail = [c for c in _pre_cols if c in df.columns]
if _pre_avail:
    vol_nonevent = df[_pre_avail].stack().std()
else:
    vol_nonevent = np.nan

print(f"\n--- Key Diagnostics ---")
if np.isfinite(vol_nonevent):
    print(f"Event-day \u03c3 vs pre-event: {vol_event:.4f} vs {vol_nonevent:.4f} "
          f"({vol_event / vol_nonevent:.1f}\u00d7)")
print(f"Mean event-day return (ret_t0): {reg['ret_t0'].mean()*10000:.1f} bps")
print(f"Mean CAR[-1,+1]:                {reg['CAR_m1_p1'].mean()*10000:.1f} bps")
print(f"Mean CAR[+2,+5]:                {reg['CAR_p2_p5'].mean()*10000:.1f} bps")

### 11.2 CAR Path by Novelty Quintile

Classic event-study visualization: cumulative returns in [-15, +15] trading
days, split by novelty quintile. H1 predicts Q5 shows larger initial reaction;
H2 predicts subsequent reversal in [+2, +5].

In [ ]:
# ── CAR path from daily returns in FINAL.csv ─────────────────────────
# Merge quintile assignments onto the event-level df (which has ret_t-15..ret_t+15)
RET_PATH_COLS = [f"ret_t{j}" for j in range(-15, 16)]
_avail = [c for c in RET_PATH_COLS if c in df.columns]

df_q = df[["ticker", "quarter_str"] + _avail].merge(
    fqs[["ticker", "quarter_str", "novelty_quintile"]],
    on=["ticker", "quarter_str"], how="inner",
)
df_q = df_q.dropna(subset=["novelty_quintile"])

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
colors_q = {1: "#34d399", 2: "#60a5fa", 3: "#f5a623", 4: "#fb923c", 5: "#f87171"}

# ── Left: CAR path [-15, +15] by quintile ────────────────────────────
for q in sorted(df_q["novelty_quintile"].unique()):
    q = int(q)
    subset = df_q[df_q["novelty_quintile"] == q]
    # Mean daily return at each tau, then cumulate
    mean_rets = subset[_avail].mean()
    car_path = mean_rets.cumsum() * 100   # convert to %
    taus = [int(c.replace("ret_t", "")) for c in _avail]
    axes[0].plot(taus, car_path.values,
                 color=colors_q[q], linewidth=2.2, label=f"Q{q}")

axes[0].axvline(0, color="white", lw=0.8, ls="--", alpha=0.4)
axes[0].axhline(0, color="white", lw=0.5, alpha=0.3)
axes[0].set_xlabel("Trading Days Relative to Earnings Call")
axes[0].set_ylabel("Cumulative Return (%)")
axes[0].set_title("Return Path by Novelty Quintile  [-15, +15]", fontsize=13)
axes[0].legend(title="Quintile", fontsize=9)

# ── Right: CAR[-1,+1] bar chart by quintile ──────────────────────────
q_car = (reg.groupby("novelty_quintile")["CAR_m1_p1"]
         .agg(["mean", "sem", "count"]).reset_index())
axes[1].bar(
    q_car["novelty_quintile"].astype(int), q_car["mean"] * 100,
    color=[colors_q[int(q)] for q in q_car["novelty_quintile"]],
    alpha=0.85, edgecolor="none",
)
axes[1].errorbar(
    q_car["novelty_quintile"].astype(int), q_car["mean"] * 100,
    yerr=q_car["sem"] * 100 * 1.96,
    fmt="none", color="white", capsize=5, lw=1.5,
)
axes[1].axhline(0, color="white", lw=0.8, ls="--", alpha=0.4)
axes[1].set_xlabel("Novelty Quintile")
axes[1].set_ylabel("Mean CAR[-1,+1] (%)")
axes[1].set_title("3-Day Announcement CAR by Quintile", fontsize=13)
axes[1].set_xticks([1, 2, 3, 4, 5])

# Styling
fig.patch.set_facecolor("#0d0d14")
for ax in axes:
    ax.set_facecolor("#0d0d14")
    for spine in ax.spines.values():
        spine.set_color("#2a2a42")
    ax.tick_params(colors="#8888aa")
    ax.xaxis.label.set_color("#e8e8f0")
    ax.yaxis.label.set_color("#e8e8f0")
    ax.title.set_color("#e8e8f0")
    if ax.get_legend():
        ax.get_legend().get_title().set_color("#e8e8f0")
        for t in ax.get_legend().get_texts():
            t.set_color("#e8e8f0")

plt.tight_layout()
plt.savefig("event_study_car.png", dpi=150, bbox_inches="tight", facecolor="#0d0d14")
plt.show()

# ── Monotonicity test ────────────────────────────────────────────────
from scipy.stats import ttest_ind
q1_cars = reg[reg["novelty_quintile"] == 1]["CAR_m1_p1"]
q5_cars = reg[reg["novelty_quintile"] == 5]["CAR_m1_p1"]
t_stat, p_val = ttest_ind(q1_cars, q5_cars, equal_var=False)
print(f"Q1 vs Q5 CAR[-1,+1]:  t = {t_stat:.3f},  p = {p_val:.4f}")
print(f"  Q1 mean: {q1_cars.mean()*100:.3f}%   Q5 mean: {q5_cars.mean()*100:.3f}%   "
      f"Spread: {(q1_cars.mean()-q5_cars.mean())*100:.3f}%")

# Same for post-event drift
q1_p = reg[reg["novelty_quintile"] == 1]["CAR_p2_p5"]
q5_p = reg[reg["novelty_quintile"] == 5]["CAR_p2_p5"]
t2, p2 = ttest_ind(q1_p, q5_p, equal_var=False)
print(f"\nQ1 vs Q5 CAR[+2,+5]:  t = {t2:.3f},  p = {p2:.4f}")
print(f"  Q1 mean: {q1_p.mean()*100:.3f}%   Q5 mean: {q5_p.mean()*100:.3f}%   "
      f"Spread: {(q1_p.mean()-q5_p.mean())*100:.3f}%")

### 11.3 Cross-Sectional Regressions (M1--M4) — PDF §7

Four specifications from the research memo. All use OLS with standard errors
**clustered by calendar quarter** (cross-sectionally correlated events).

| Model | DV | Key Regressors | Hypothesis |
|-------|----|----------------|------------|
| **M1** | CAR[-1,+1] | Q2, Q3, Q4, Q5 dummies | H1: Overreaction (beta on Q5) |
| **M2** | CAR[-1,+1] | NovScore, NovScore^2 | Non-linear novelty-return |
| **M3** | CAR[+2,+5] | Q5, CAR[-1,+1], Q5 x CAR[-1,+1] | H2: Mean reversion (beta3 < 0) |
| **M4** | CAR[-1,+1] | NovScore, abs(pre_drift), interaction | Pre-priced novelty (beta3 < 0) |

**Controls**: SUE, close_to_open, log(word_count), pre_drift, Quarter FE, Year FE.
Note: log(mktcap), book_to_market, analyst_coverage not yet merged into FINAL.csv (see §10 gaps).

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# M1  \u00b7  Quintile Sort (main)   — PDF §7
# CAR[-1,+1] = \u03b1 + \u03b2\u2081Q2 + \u03b2\u2082Q3 + \u03b2\u2083Q4 + \u03b2\u2084Q5
#            + \u03b3SUE + \u03b3CtO + \u03b3log(wc) + \u03b3pre_drift + QtrFE + YrFE
# ══════════════════════════════════════════════════════════════════════
m1 = nh.fit_ols_cluster(
    "CAR_m1_p1 ~ Q2 + Q3 + Q4 + Q5"
    " + sue + close_to_open_return + log_wc + pre_drift"
    " + C(qtr_fe) + C(yr_fe)",
    data=reg,
    cluster_col="quarter_str",
)
print("\u2550" * 60)
print("M1  \u00b7  Quintile Sort   (DV = CAR[-1,+1])")
print("\u2550" * 60)
_tbl = m1.summary2().tables[1][["Coef.", "Std.Err.", "t", "P>|t|"]]
print(_tbl.loc[[c for c in _tbl.index if not c.startswith("C(")]].to_string())
print(f"\nN = {int(m1.nobs):,}   R\u00b2 = {m1.rsquared:.4f}")
print(f"\u03b2(Q5) = {m1.params['Q5']:.5f}  "
      f"(t = {m1.tvalues['Q5']:.2f}, p = {m1.pvalues['Q5']:.4f})")

# ══════════════════════════════════════════════════════════════════════
# M2  \u00b7  Continuous + Quadratic
# CAR[-1,+1] = \u03b1 + \u03b2\u2081NovScore + \u03b2\u2082NovScore\u00b2 + controls
# ══════════════════════════════════════════════════════════════════════
m2 = nh.fit_ols_cluster(
    "CAR_m1_p1 ~ nov_score + nov_score_sq"
    " + sue + close_to_open_return + log_wc + pre_drift"
    " + C(qtr_fe) + C(yr_fe)",
    data=reg,
    cluster_col="quarter_str",
)
print("\n" + "\u2550" * 60)
print("M2  \u00b7  Continuous + Quadratic   (DV = CAR[-1,+1])")
print("\u2550" * 60)
_tbl2 = m2.summary2().tables[1][["Coef.", "Std.Err.", "t", "P>|t|"]]
print(_tbl2.loc[[c for c in _tbl2.index if not c.startswith("C(")]].to_string())
print(f"\nN = {int(m2.nobs):,}   R\u00b2 = {m2.rsquared:.4f}")
print(f"\u03b2(NovScore)  = {m2.params['nov_score']:.5f}  "
      f"(t = {m2.tvalues['nov_score']:.2f})")
print(f"\u03b2(NovScore\u00b2) = {m2.params['nov_score_sq']:.5f}  "
      f"(t = {m2.tvalues['nov_score_sq']:.2f})")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# M3  \u00b7  Mean Reversion (H2 test)
# CAR[+2,+5] = \u03b1 + \u03b2\u2081Q5 + \u03b2\u2082CAR[-1,+1]
#            + \u03b2\u2083(Q5 \u00d7 CAR[-1,+1]) + controls
# ══════════════════════════════════════════════════════════════════════
m3 = nh.fit_ols_cluster(
    "CAR_p2_p5 ~ Q5 + CAR_m1_p1 + Q5_x_car"
    " + sue + close_to_open_return + log_wc + pre_drift"
    " + C(qtr_fe) + C(yr_fe)",
    data=reg,
    cluster_col="quarter_str",
)
print("\u2550" * 60)
print("M3  \u00b7  Mean Reversion   (DV = CAR[+2,+5])")
print("\u2550" * 60)
_tbl3 = m3.summary2().tables[1][["Coef.", "Std.Err.", "t", "P>|t|"]]
print(_tbl3.loc[[c for c in _tbl3.index if not c.startswith("C(")]].to_string())
print(f"\nN = {int(m3.nobs):,}   R\u00b2 = {m3.rsquared:.4f}")
print(f"\u03b2\u2083(Q5 \u00d7 CAR[-1,+1]) = {m3.params['Q5_x_car']:.5f}  "
      f"(t = {m3.tvalues['Q5_x_car']:.2f}, p = {m3.pvalues['Q5_x_car']:.4f})")
print("\u03b2\u2083 < 0 \u27f9 high-novelty firms with large initial moves "
      "show stronger reversal")

# ══════════════════════════════════════════════════════════════════════
# M4  \u00b7  Pre-Priced Novelty
# CAR[-1,+1] = \u03b1 + \u03b2\u2081NovScore + \u03b2\u2082|pre_drift|
#            + \u03b2\u2083(NovScore \u00d7 |pre_drift|) + controls
# ══════════════════════════════════════════════════════════════════════
m4 = nh.fit_ols_cluster(
    "CAR_m1_p1 ~ nov_score + abs_pre_drift + nov_x_abs_pre"
    " + sue + close_to_open_return + log_wc"
    " + C(qtr_fe) + C(yr_fe)",
    data=reg,
    cluster_col="quarter_str",
)
print("\n" + "\u2550" * 60)
print("M4  \u00b7  Pre-Priced Novelty   (DV = CAR[-1,+1])")
print("\u2550" * 60)
_tbl4 = m4.summary2().tables[1][["Coef.", "Std.Err.", "t", "P>|t|"]]
print(_tbl4.loc[[c for c in _tbl4.index if not c.startswith("C(")]].to_string())
print(f"\nN = {int(m4.nobs):,}   R\u00b2 = {m4.rsquared:.4f}")
print(f"\u03b2\u2083(Nov \u00d7 |pre_drift|) = {m4.params['nov_x_abs_pre']:.5f}  "
      f"(t = {m4.tvalues['nov_x_abs_pre']:.2f}, "
      f"p = {m4.pvalues['nov_x_abs_pre']:.4f})")
print("\u03b2\u2083 < 0 \u27f9 novelty effect attenuated when market has "
      "already moved pre-call")

In [ ]:
# ── Combined Regression Summary Table ────────────────────────────────
def _extract(model, var, label=None):
    label = label or var
    if var in model.params.index:
        return {
            "Variable": label,
            "Coef":   f"{model.params[var]:.5f}",
            "t-stat": f"{model.tvalues[var]:.2f}",
            "p-val":  f"{model.pvalues[var]:.4f}",
        }
    return {"Variable": label, "Coef": "", "t-stat": "", "p-val": ""}

summary_rows = []
for name, model, keys in [
    ("M1", m1, [("Q5", "Q5 (highest novelty)")]),
    ("M2", m2, [("nov_score", "NovScore"), ("nov_score_sq", "NovScore\u00b2")]),
    ("M3", m3, [("Q5", "Q5"), ("CAR_m1_p1", "CAR[-1,+1]"),
                ("Q5_x_car", "Q5 \u00d7 CAR[-1,+1]")]),
    ("M4", m4, [("nov_score", "NovScore"), ("abs_pre_drift", "|pre_drift|"),
                ("nov_x_abs_pre", "Nov \u00d7 |pre_drift|")]),
]:
    for var, label in keys:
        row = _extract(model, var, label)
        row["Model"] = name
        row["N"]     = f"{int(model.nobs):,}"
        row["R\u00b2"]  = f"{model.rsquared:.4f}"
        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)[
    ["Model", "Variable", "Coef", "t-stat", "p-val", "N", "R\u00b2"]
]
print("\n" + "\u2550" * 72)
print("REGRESSION SUMMARY  (SEs clustered by calendar quarter)")
print("\u2550" * 72)
print(summary_df.to_string(index=False))

### 11.4 Fama--French 5-Factor Calendar-Time Alpha — PDF §6.1

Portfolio construction per the memo:
- **Ranking**: end of calendar quarter t, using adjusted novelty from current-quarter transcripts
- **Holding**: quarter t+1, equal-weighted within each quintile
- **Long-short**: R_LS = R_Q1 (familiar) -- R_Q5 (novel)
- **Risk model**: R_LS -- R_f = alpha + b1*MKT + b2*SMB + b3*HML + b4*RMW + b5*CMA + eps

Uses `novelty_helpers.ff5_alpha_regression()` with monthly factors from Ken French's website.

In [ ]:
# ── Calendar-time L-S portfolio: monthly returns ─────────────────────
# For each firm-quarter, use the daily returns in the POST-event window
# (ret_t0 .. ret_t15) to compute a holding-period return, then aggregate
# to monthly equal-weighted quintile portfolios.

# Map each event to its calendar month
_event = df[["ticker", "quarter_str", "mostimportantdateutc"]].drop_duplicates(
    subset=["ticker", "quarter_str"], keep="last").copy()
_event["event_date"] = pd.to_datetime(_event["mostimportantdateutc"], errors="coerce")
_event["event_ym"] = _event["event_date"].dt.to_period("M")

# Merge quintile + holding-period daily returns
_hold_cols = [f"ret_t{j}" for j in range(0, 16) if f"ret_t{j}" in df.columns]
_hold = df[["ticker", "quarter_str"] + _hold_cols].merge(
    fqs[["ticker", "quarter_str", "novelty_quintile"]],
    on=["ticker", "quarter_str"], how="inner",
).merge(_event[["ticker", "quarter_str", "event_ym"]], on=["ticker", "quarter_str"])
_hold = _hold.dropna(subset=["novelty_quintile", "event_ym"])

# Compound daily returns into a single holding return per firm-event
_hold["hold_ret"] = (1 + _hold[_hold_cols].fillna(0)).prod(axis=1) - 1

# Equal-weighted monthly portfolio returns by quintile
port_monthly = (_hold
    .groupby(["event_ym", "novelty_quintile"])["hold_ret"]
    .mean()
    .unstack("novelty_quintile"))
port_monthly.columns = [f"Q{int(c)}" for c in port_monthly.columns]

if "Q1" in port_monthly.columns and "Q5" in port_monthly.columns:
    port_monthly["L_S"] = port_monthly["Q1"] - port_monthly["Q5"]
else:
    raise ValueError("Need both Q1 and Q5 for long-short portfolio")

# Convert period index to timestamp for ff5 merge
ls_monthly = port_monthly["L_S"].copy()
ls_monthly.index = ls_monthly.index.to_timestamp()

print(f"Portfolio months: {len(port_monthly)}")
print(f"Mean monthly L-S return: {ls_monthly.mean()*100:.3f}%")
ann_sharpe = ls_monthly.mean() / ls_monthly.std() * np.sqrt(12)
print(f"Annualized Sharpe: {ann_sharpe:.2f}")

In [ ]:
# ── FF5 alpha regression ─────────────────────────────────────────────
try:
    date_lo = ls_monthly.index.min().strftime("%Y-%m-%d")
    date_hi = ls_monthly.index.max().strftime("%Y-%m-%d")
    ff5_fit = nh.ff5_alpha_regression(
        ls_monthly,
        factors=nh.fetch_ff5_monthly(start=date_lo, end=date_hi),
        excess_returns=False,   # ls_monthly is raw L-S, not excess
    )

    print("\u2550" * 60)
    print("Fama\u2013French 5-Factor Alpha  (Monthly, L\u2013S = Q1 \u2212 Q5)")
    print("\u2550" * 60)
    print(ff5_fit.summary2().tables[1].to_string())
    print(f"\nAlpha (monthly):    {ff5_fit.params['const']*100:.3f}%  "
          f"(t = {ff5_fit.tvalues['const']:.2f})")
    print(f"Alpha (annualized): {ff5_fit.params['const']*12*100:.1f}% "
          f"= {ff5_fit.params['const']*12*10000:.0f} bps/yr")
    print(f"N months: {int(ff5_fit.nobs)}")
except Exception as ex:
    print(f"FF5 regression failed: {ex}")
    print("Check network access for Ken French factor download.")

In [ ]:
# ── Cumulative long-short portfolio returns ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: cumulative L-S return
cum_ls = (1 + port_monthly["L_S"]).cumprod() - 1
x_dates = cum_ls.index.to_timestamp()
axes[0].plot(x_dates, cum_ls.values * 100, color="#f5a623", linewidth=2)
axes[0].axhline(0, color="white", lw=0.6, ls="--", alpha=0.3)
axes[0].fill_between(x_dates, 0, cum_ls.values * 100,
                     where=cum_ls.values > 0, color="#34d399", alpha=0.15)
axes[0].fill_between(x_dates, 0, cum_ls.values * 100,
                     where=cum_ls.values < 0, color="#f87171", alpha=0.15)
axes[0].set_ylabel("Cumulative Return (%)")
axes[0].set_title("Cumulative L-S Portfolio  (Q1 \u2212 Q5)", fontsize=13)

# Right: monthly L-S bar chart
bar_x = port_monthly.index.to_timestamp()
bar_colors = np.where(port_monthly["L_S"] > 0, "#34d399", "#f87171")
axes[1].bar(bar_x, port_monthly["L_S"].values * 100,
            width=25, color=bar_colors, alpha=0.7, edgecolor="none")
axes[1].axhline(0, color="white", lw=0.6, ls="--", alpha=0.3)
axes[1].set_ylabel("Monthly Return (%)")
axes[1].set_title("Monthly L-S Returns", fontsize=13)

fig.patch.set_facecolor("#0d0d14")
for ax in axes:
    ax.set_facecolor("#0d0d14")
    for spine in ax.spines.values():
        spine.set_color("#2a2a42")
    ax.tick_params(colors="#8888aa")
    ax.xaxis.label.set_color("#e8e8f0")
    ax.yaxis.label.set_color("#e8e8f0")
    ax.title.set_color("#e8e8f0")

plt.tight_layout()
plt.savefig("ff5_portfolio.png", dpi=150, bbox_inches="tight", facecolor="#0d0d14")
plt.show()